# 🕸️ 06 · Graphs and trees: networks and phylogenies
### Graphs as tuples ⟨V, E⟩, directed and undirected edges, paths, edge labels and weights, neighbourhoods and degrees, regular and complete graphs, cliques, subgraphs and induced subgraphs, bipartite graphs, connectivity and breadth-first search, trees, rooted and k-ary trees, phylogenies and spanning trees — with gene-regulatory, protein-interaction, metabolic and co-expression networks

**BiologyToMath** · Part B — Foundations: logic, sets, functions, sequences and graphs · Module 06 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/06_graphs_and_trees.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib, SciPy (`cluster.hierarchy` to check UPGMA); networkx only as an optional cross-check |
| **Prerequisites** | [Module 02 · 🔣 Logic: the grammar of precise reasoning](02_logic.ipynb) (∀, ∃, negation), [Module 03 · 🧺 Sets and functions: collections and mappings](03_sets_and_functions.ipynb) (sets, power sets, functions, indicator functions), [Module 04 · 🔤 Tuples, sequences and strings: when order matters](04_tuples_sequences_strings.ipynb) (tuples, Cartesian products, relations, Σ*, k-mers) and [Module 05 · ⏱️ Asymptotics: how fast functions grow](05_asymptotics_growth_of_functions.ipynb) (Big-O, counting trees). |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Graphs: G = ⟨V, E⟩, nodes/vertices, edges/links, networks; undirected (symmetric) vs directed graphs, E ⊆ V × V vs E ⊆ {{u, v} : u, v ∈ V}, self-loops; drawing conventions; directed graphs as Bayesian-network skeletons (adjacent, parent, child, directed path); paths ⟨v₁, …, v_k⟩ and their length; edge-label and weight functions ℓ: E → Σ*, w: E → ℝ, path labels and weights; neighbourhoods N_G(v) and degree with the indicator function (self-loops), the "loops count twice" convention and the degree exercises; in-/out-neighbourhoods and in-/out-degree (flight network, the A–H example); regular and d-regular graphs; complete graphs Kₙ and cliques; subgraphs and induced subgraphs (all 3-node subgraphs; a regulatory network restricted to differentially expressed genes); bipartite graphs (isolated nodes; which of five graphs are bipartite); connectivity (∀∃ definition) and connected components; trees (connected with |V| − 1 edges, no cycles), leaves and internal nodes, rooted trees (root, parent, children, levels, subtrees), k-ary and full k-ary trees (T1–T4), undirected/directed trees and polytrees; phylogenies (human, chimpanzee, mouse); spanning trees |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. The inflammation network is a simplified, illustrative cartoon of innate-immune signalling (toolkit `INFLAMMATION_NETWORK_EDGES`), not a curated database. Expression data are **SIMULATED** (`simulate_tissue_expression`); the degree-distribution networks and the DNA for the de Bruijn graph are SIMULATED too. The protein-interaction, metabolic, route-graph distances and species-distance examples are small typed-in tables; distances and branch lengths are illustrative, not measured values.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Define** a graph as a tuple $G = \langle V, E\rangle$, **distinguish** directed ($E\subseteq V\times V$) from undirected edges (2-element sets), and **choose** the right kind for a biological network.
2. **Recognise** paths and **compute** path lengths, path labels (concatenation in $\Sigma^*$) and path weights (sums) from edge-label and weight functions.
3. **Compute** neighbourhoods and degrees — with the indicator-function rule for self-loops, in- and out-degrees in directed graphs — and **prove** and **use** the handshake lemma $\sum_v\deg(v) = 2|E|$.
4. **Identify** regular graphs, complete graphs $K_n$ (with $\binom n2$ edges), cliques, subgraphs and induced subgraphs, and **extract** the regulatory subnetwork of genes changed by infection.
5. **Test** whether a graph is bipartite with a two-colouring algorithm, and **model** metabolism as a bipartite graph of metabolites and reactions.
6. **Implement** breadth-first search to find connected components, shortest path lengths and reachable genes.
7. **Characterise** trees (connected, $|V|-1$ edges, no cycles), rooted trees (root, parents, children, levels, leaves) and full $k$-ary trees, and **read** a phylogeny.
8. **Build** a spanning tree by BFS, a minimum spanning tree by Kruskal's algorithm and a small UPGMA phylogeny, and **connect** them to single-linkage clustering, adjacency matrices and de Bruijn assembly graphs.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the inflammation network and the infection data | 5 min |
| 1 · What is a graph? | ⟨V, E⟩; directed vs undirected; self-loops; drawing | 15 min |
| 2 · Directed graphs and Bayesian networks | adjacent, parent, child, directed path | 10 min |
| 3 · Paths, labels and weights | course schedules, a route graph, a genome graph | 15 min |
| 4 · Neighbourhoods and degrees | indicator functions, loops, handshake lemma, the degree exercises, hubs | 20 min |
| 5 · Directed degrees, regular and complete graphs | in/out-degree, the A–H example, feedback loops, $K_n$, cliques | 20 min |
| 6 · Subgraphs | all 3-node subgraphs; the subnetwork of infection-responsive genes | 15 min |
| 7 · Bipartite graphs | two-colouring; which of five graphs are bipartite; metabolism | 15 min |
| 8 · Connectivity and breadth-first search | components, distances, reachability, co-expression modules | 15 min |
| 9 · Trees | leaves, rooted trees, k-ary trees T1–T4, polytrees | 20 min |
| 10 · Phylogenies and spanning trees | human–chimp–mouse, UPGMA, BFS trees, Kruskal, single linkage | 15 min |
| 11 · Graphs as matrices | adjacency matrix, $A^2$ counts 2-step walks, de Bruijn graphs | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

When bacteria invade a tissue, immune cells sense them and **hundreds of genes change expression within hours**. A list
of changed genes is a *set* (Module 03) — but the cell does not experience a set. It experiences a **network**: the
transcription factor NF-κB switches on the cytokines TNF and IL-1β, which in turn activate NF-κB again (a feedback
loop); IL-6 activates STAT3, which induces acute-phase proteins in the liver; STAT1 induces interferon-stimulated genes.
Which genes are the *regulators* and which are *targets*? Which feedback loops can sustain inflammation? Which changed
genes are wired together, and which regulators are hidden because their activity changes without their mRNA level
changing?

Biology is full of such relationships: proteins that bind each other, metabolites converted by reactions, species that
share ancestors. The mathematics of relationships between pairs of objects is **graph theory**, and trees — the graphs
of phylogenies — are its most famous special case.

| Biological question | Mathematical tool | Where in this module |
|---|---|---|
| Who regulates whom? Is "binds" different from "activates"? | directed vs undirected graphs | Sections 1–2 |
| Which cascades lead from a receptor to a gene? | paths, labels, weights | Section 3 |
| Which proteins are hubs of the interaction network? | degree, handshake lemma | Section 4 |
| Which regulators drive the response, and which loops feed back? | in/out-degree, cycles | Section 5 |
| How are the genes changed by infection wired together? | induced subgraphs | Section 6 |
| How are metabolites and reactions connected? | bipartite graphs | Section 7 |
| Which genes can NF-κB influence? Which genes form co-expression modules? | BFS, connected components | Section 8 |
| How are human, chimpanzee and mouse related? | rooted trees, phylogenies | Sections 9–10 |
| How do we build a tree from a distance matrix? | UPGMA, minimum spanning trees | Section 10 |
| How many regulatory paths of length 2 exist? How is a genome assembled? | adjacency matrix, de Bruijn graphs | Section 11 |

> 🧪 **Simulated data, with a known truth.** The network is the toolkit's small, illustrative cartoon of innate-immune
> signalling (15 regulator → target edges among 14 real genes). The expression data come from
> `simulate_tissue_expression`, whose hidden truth includes exactly which genes respond to infection in spleen, kidney
> and liver — so we can check what a network analysis recovers.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import itertools
import math
from collections import Counter, deque

from scipy.cluster import hierarchy

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). They draw small graphs with Matplotlib only, lay out circles and trees, build adjacency matrices, provide the illustrative inflammation network, simulate the infection experiment and DNA, and count k-mers. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def circle_layout(nodes: list) -> dict:
    """Place nodes evenly on a circle (first node at the top, clockwise)."""
    n = len(nodes)
    angles = np.pi / 2 - 2 * np.pi * np.arange(n) / max(n, 1)
    return {v: (float(np.cos(a)), float(np.sin(a))) for v, a in zip(nodes, angles)}


def tree_layout(edges: list, root) -> dict:
    """Positions for drawing a rooted tree top-down: root at the top, leaves evenly spaced.

    ``edges`` are (parent, child) pairs or undirected pairs; the tree is explored from ``root``.
    """
    nbrs = {}
    for u, v in edges:
        nbrs.setdefault(u, []).append(v)
        nbrs.setdefault(v, []).append(u)
    children, order, seen, stack = {}, [], {root}, [root]
    while stack:
        u = stack.pop()
        order.append(u)
        children[u] = [w for w in nbrs.get(u, []) if w not in seen]
        seen.update(children[u])
        stack.extend(reversed(children[u]))
    depth = {root: 0}
    for u in order:
        for w in children[u]:
            depth[w] = depth[u] + 1
    x_pos, next_leaf = {}, [0]

    def place(u):
        if not children[u]:
            x_pos[u] = float(next_leaf[0])
            next_leaf[0] += 1
        else:
            for w in children[u]:
                place(w)
            x_pos[u] = float(np.mean([x_pos[w] for w in children[u]]))

    place(root)
    return {u: (x_pos[u], -float(depth[u])) for u in order}


def adjacency_matrix(nodes: list, edges: list, directed: bool = False) -> np.ndarray:
    """Adjacency matrix A with A[i, j] = number of edges from nodes[i] to nodes[j].

    For an undirected graph every edge is counted in both directions (A is symmetric);
    a self-loop {v, v} adds 1 to the diagonal entry A[v, v].  List each undirected edge once:
    giving both (u, v) and (v, u) with ``directed=False`` counts that edge twice (a multigraph).
    """
    index = {v: i for i, v in enumerate(nodes)}
    A = np.zeros((len(nodes), len(nodes)), dtype=int)
    for u, v in edges:
        A[index[u], index[v]] += 1
        if not directed and u != v:
            A[index[v], index[u]] += 1
    return A


def draw_graph(edges: list, nodes=None, directed: bool = False, pos=None, ax=None, node_colors=None,
               edge_labels=None, title: str | None = None, font_size: int = 9):
    """Draw a small graph with matplotlib only (no extra packages needed).

    ``edges`` is a list of (u, v) pairs; ``nodes`` (optional) adds isolated nodes and fixes the
    order; ``pos`` maps node → (x, y) (default: ``circle_layout``).  Directed edges get arrows,
    self-loops are drawn as small rings above the node, ``edge_labels`` maps (u, v) → text (e.g. a
    weight) and ``node_colors`` maps node → colour.  Node circles grow with the length of their
    label.  Returns the Axes.
    """
    from matplotlib.transforms import offset_copy

    if nodes is None:
        nodes = []
        for u, v in edges:
            for w in (u, v):
                if w not in nodes:
                    nodes.append(w)
    pos = circle_layout(list(nodes)) if pos is None else pos
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 3.6))
    radius = {v: max(0.62 * font_size * len(str(v)) / 2 + 4, 0.9 * font_size) for v in nodes}  # points
    for u, v in edges:
        if u == v:
            ring = offset_copy(ax.transData, fig=ax.figure, x=0, y=1.25 * radius[u], units="points")
            ax.scatter([pos[u][0]], [pos[u][1]], s=(1.3 * radius[u]) ** 2, facecolors="none",
                       edgecolors="0.35", linewidths=1.3, transform=ring, zorder=1)
            continue
        style = "-|>" if directed else "-"
        ax.annotate("", xy=pos[v], xytext=pos[u],
                    arrowprops=dict(arrowstyle=style, color="0.35", lw=1.3, mutation_scale=12,
                                    shrinkA=radius[u], shrinkB=radius[v],
                                    connectionstyle="arc3,rad=0.08" if directed else "arc3,rad=0"))
        if edge_labels and (u, v) in edge_labels:
            (x1, y1), (x2, y2) = pos[u], pos[v]
            ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(edge_labels[(u, v)]), fontsize=font_size - 1,
                    ha="center", va="center", color="darkred", zorder=4,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
    xs = [pos[v][0] for v in nodes]
    ys = [pos[v][1] for v in nodes]
    colours = [(node_colors or {}).get(v, "#cfe2f3") for v in nodes]
    ax.scatter(xs, ys, s=[(2 * radius[v]) ** 2 for v in nodes], c=colours, edgecolors="0.2",
               linewidths=1.2, zorder=2, clip_on=False)
    for v in nodes:
        ax.text(pos[v][0], pos[v][1], str(v), ha="center", va="center", fontsize=font_size, zorder=3,
                clip_on=False)
    ax.margins(0.22)
    ax.set_aspect("equal", adjustable="datalim")
    ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


NUCLEOTIDES = ("A", "C", "G", "T")


def simulate_dna(length: int, transitions=None, initial=None, seed: int = 0) -> str:
    """Random DNA string.  Without ``transitions`` the bases are i.i.d. uniform; otherwise they
    follow a first-order Markov chain with the given 4 × 4 transition matrix (order A, C, G, T)
    and initial distribution ``initial`` (default uniform)."""
    rng = np.random.default_rng(seed)
    initial = np.full(4, 0.25) if initial is None else np.asarray(initial, dtype=float)
    if transitions is None:
        return "".join(rng.choice(list(NUCLEOTIDES), size=length, p=initial))
    P = np.asarray(transitions, dtype=float)
    states = [rng.choice(4, p=initial)]
    for _ in range(length - 1):
        states.append(rng.choice(4, p=P[states[-1]]))
    return "".join(NUCLEOTIDES[s] for s in states)


def kmer_counts(seq: str, k: int) -> dict:
    """Counts of every overlapping k-mer (substring of length k) of ``seq``, sorted alphabetically."""
    counts = Counter(seq[i:i + k] for i in range(len(seq) - k + 1))
    return dict(sorted(counts.items()))


# Named genes of the simulated infection experiment.  The infection-response genes are real genes
# known to be induced during infection (cytokines, interferon-stimulated genes, acute-phase
# proteins; lysozyme, LYZ, is an antimicrobial enzyme rather than a classic acute-phase protein and is
# grouped with them only for simplicity); the reference genes are classic "housekeeping" genes used
# to normalise qPCR.
INFECTION_PATHWAYS = {
    "cytokine signalling": ["IL1B", "TNF", "IL6", "CXCL8"],
    "interferon response": ["IRF1", "STAT1", "MX1", "ISG15"],
    "acute phase": ["SAA1", "HAMP", "C3", "LYZ"],
}


HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "EEF1A1", "RPL13A", "TBP", "HPRT1",
                      "PPIA", "UBC", "YWHAZ", "SDHA", "TUBA1A", "B2M"]


# A simplified, illustrative cartoon of innate-immune signalling among the named genes
# (regulator → target).  NF-κB (NFKB1) induces cytokines; TNF and IL-1β activate NF-κB (a feedback
# loop); IL-6 acts through STAT3 to induce acute-phase genes; STAT1 induces interferon-stimulated genes.
INFLAMMATION_NETWORK_EDGES = [
    ("NFKB1", "IL1B"), ("NFKB1", "TNF"), ("NFKB1", "IL6"), ("NFKB1", "CXCL8"),
    ("TNF", "NFKB1"), ("IL1B", "NFKB1"), ("IL6", "STAT3"), ("STAT3", "SAA1"),
    ("STAT3", "HAMP"), ("STAT3", "C3"), ("STAT1", "IRF1"), ("STAT1", "MX1"),
    ("STAT1", "ISG15"), ("IRF1", "ISG15"), ("NFKB1", "LYZ"),
]


def simulate_tissue_expression(n_genes: int = 400, tissues=("spleen", "kidney", "liver"), n_reps: int = 4,
                               seed: int = 2680) -> dict:
    """SIMULATED log2 gene expression from an infection experiment in several tissues.

    Every tissue has ``n_reps`` control and ``n_reps`` infected samples.  The 12 genes of
    ``INFECTION_PATHWAYS`` respond to infection in every tissue (up by 1.5–3 log2 units); further
    "g…" genes respond in one tissue only or in two tissues, so the sets of truly changed genes
    overlap like the differentially-expressed-gene sets of a real multi-tissue study.  The 12
    ``HOUSEKEEPING_GENES`` are stable (tiny noise, no response).  About a third of all genes differ in
    baseline level between tissues.

    Returns a dict with ``X`` (samples × genes, log2 expression), ``samples`` (DataFrame with
    columns sample, tissue, condition), ``genes`` (list), ``true_lfc`` (DataFrame tissues × genes,
    true infected − control difference), ``true_de`` (dict tissue → set of truly changed genes),
    ``pathways`` (dict pathway → genes) and ``source``.
    """
    rng = np.random.default_rng(seed)
    named = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
    n_other = n_genes - len(named) - len(HOUSEKEEPING_GENES)
    if n_other < 60:
        raise ValueError("n_genes must be at least 84")
    genes = named + HOUSEKEEPING_GENES + [f"g{i:03d}" for i in range(1, n_other + 1)]
    J, T = len(genes), len(tissues)
    base = rng.normal(7.0, 1.8, J)
    base[len(named):len(named) + len(HOUSEKEEPING_GENES)] = rng.uniform(9.0, 12.0, len(HOUSEKEEPING_GENES))
    tissue_shift = rng.normal(0.0, 1.0, (T, J)) * (rng.random(J) < 0.35)
    tissue_shift[:, len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.0
    lfc = np.zeros((T, J))
    lfc[:, :len(named)] = rng.uniform(1.5, 3.0, (T, len(named)))
    other = list(range(len(named) + len(HOUSEKEEPING_GENES), J))
    rng.shuffle(other)
    sizes_single = [24, 18, 14, 12, 10][:T]
    pairs = list(itertools.combinations(range(T), 2))
    sizes_pair = [10, 5, 4, 3, 3, 2, 2, 2, 2, 2][:len(pairs)]
    k = 0

    def effect(m):
        return rng.uniform(1.0, 2.5, m) * np.where(rng.random(m) < 0.7, 1.0, -1.0)

    for t, m in enumerate(sizes_single):
        idx = other[k:k + m]
        lfc[t, idx] = effect(m)
        k += m
    for (t1, t2), m in zip(pairs, sizes_pair):
        idx = other[k:k + m]
        e = effect(m)
        lfc[t1, idx] = e
        lfc[t2, idx] = e * rng.uniform(0.8, 1.2, m)
        k += m
    noise_sd = rng.uniform(0.25, 0.55, J)
    noise_sd[len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.08
    rows, X = [], []
    for t, tissue in enumerate(tissues):
        for condition in ("control", "infected"):
            for r in range(1, n_reps + 1):
                mean = base + tissue_shift[t] + (lfc[t] if condition == "infected" else 0.0)
                X.append(mean + rng.normal(0.0, noise_sd))
                rows.append((f"{tissue}_{condition[:3]}{r}", tissue, condition))
    samples = pd.DataFrame(rows, columns=["sample", "tissue", "condition"])
    true_lfc = pd.DataFrame(lfc, index=list(tissues), columns=genes)
    true_de = {tissue: {g for g, v in zip(genes, lfc[t]) if v != 0} for t, tissue in enumerate(tissues)}
    return {"X": np.array(X), "samples": samples, "genes": genes, "true_lfc": true_lfc,
            "true_de": true_de, "pathways": {k_: list(v) for k_, v in INFECTION_PATHWAYS.items()},
            "source": "SIMULATED log2 expression (BiologyToMath simulate_tissue_expression)"}

print("🧰 toolkit loaded:", ", ".join(sorted(['HOUSEKEEPING_GENES', 'INFECTION_PATHWAYS', 'INFLAMMATION_NETWORK_EDGES', 'NUCLEOTIDES', 'adjacency_matrix', 'circle_layout', 'draw_graph', 'kmer_counts', 'simulate_dna', 'simulate_tissue_expression', 'tree_layout'])))

In [ ]:
# ── 0.1 This module's data: the inflammation network (illustrative) and the SIMULATED infection experiment ──
NET_EDGES = list(INFLAMMATION_NETWORK_EDGES)                     # (regulator, target) pairs
NET_NODES = sorted({gene for edge in NET_EDGES for gene in edge})
print(f"inflammation network: |V| = {len(NET_NODES)} genes, |E| = {len(NET_EDGES)} directed edges")
print("V =", NET_NODES)

infection = simulate_tissue_expression()                        # 24 samples × 400 genes, log2 expression
print("\nexpression matrix:", infection["X"].shape, "| tissues:", sorted(infection["true_de"]))
for tissue, changed in infection["true_de"].items():
    print(f"  {tissue:6s}: {len(changed):2d} truly changed genes, of which in the network: "
          f"{len(changed & set(NET_NODES))}")
print("network genes that are not measured at all:", sorted(set(NET_NODES) - set(infection["genes"])))

# a fixed drawing position for every gene (chosen by hand so that the modules are easy to see)
NET_POS = {"NFKB1": (0.0, 0.0), "TNF": (-1.3, 1.3), "IL1B": (1.3, 1.3), "CXCL8": (-1.7, -0.5), "LYZ": (-0.8, -1.5),
           "IL6": (1.4, -0.7), "STAT3": (2.8, -0.7), "SAA1": (4.1, 0.4), "HAMP": (4.3, -0.7), "C3": (4.1, -1.8),
           "STAT1": (5.7, 0.0), "IRF1": (6.9, 1.2), "MX1": (6.9, -1.2), "ISG15": (8.0, 0.0)}

🔍 **What just happened?** The network has **14 genes and 15 directed edges**, each a pair (regulator, target). The
simulated experiment measures 400 genes in 24 samples; the hidden truth lists 51, 44 and 35 genes that respond to
infection in spleen, kidney and liver, and in every tissue 12 of them are network genes (the cytokines, the
interferon-response genes and the acute-phase genes). Two network genes, the transcription factors **NFKB1** and
**STAT3**, are not measured at all — keep that in mind for Section 6. `NET_POS` just fixes where each gene is drawn.

---
## 1 · What is a graph?

> 📍 **Where we are.** Module 04 defined relations: subsets of a Cartesian product, such as the pairs (TF, target). A
> **graph** is the same idea with a picture attached — and with the vocabulary needed to talk about *structure*:
> neighbours, paths, hubs, cycles, components. Graphs are powerful tools for modelling structured relationships
> between entities.

A graph consists of
* **nodes** (or **vertices**) — the entities (genes, proteins, people, species), and
* **edges** (or **links**) — the relationships between pairs of nodes.

A graph is also called a **network**. Formally, we write a graph as a **tuple**

$$G = \langle V, E\rangle, \qquad E\subseteq V\times V \ \text{ if } G \text{ is directed}, \qquad E\subseteq\big\{\{u, v\} : u, v\in V\big\} \ \text{ if } G \text{ is undirected}.$$

*In words:* a graph is an ordered pair whose first component is a non-empty set $V$ of nodes and whose second component
is a set $E$ of edges; in a **directed** graph each edge is an *ordered* pair ⟨u, v⟩ — "an edge **from** u **to** v" —
and in an **undirected** graph each edge is an *unordered* pair {u, v} — "an edge **between** u and v".

**Why two definitions for E?** Because edges can be ordered or unordered (Module 04: tuples vs sets).
* **Directed:** each edge $(u, v)\in E$ is a directed connection from $u$ to $v$, and the order matters:
  $(u, v)\ne(v, u)$. In a social network, following, influencing or messaging is directed — on Twitter, Alice can
  follow Bob without Bob following back.
* **Undirected:** each edge $\{u, v\}\in E$ is a mutual, symmetric relationship: if A is connected to B, then B is
  connected to A, and $\{u, v\} = \{v, u\}$. Facebook friendship exists only if both agree. Undirected graphs suit
  systems where all connections are peer-to-peer.
* **Can $u = v$?** Yes: $\{v, v\} = \{v\}$ (or $(v, v)$) is allowed and represents a **self-loop**. A gene that
  activates its own transcription has a self-loop in a regulatory network.

**Drawing conventions.** Nodes are drawn as circles and edges as lines; each edge of a directed graph is drawn with an
arrow indicating its orientation ("which way it goes"). The position of a node on the page carries no meaning — only
the sets $V$ and $E$ do.

In [ ]:
# ── 1.1 Directed edges are tuples, undirected edges are 2-element sets (frozensets in Python) ──
follows = ("Alice", "Bob")                              # Twitter: Alice follows Bob (directed)
friends = frozenset({"Alice", "Bob"})                   # Facebook: Alice and Bob are friends (undirected)
print("directed:   (Alice, Bob) == (Bob, Alice)?", follows == ("Bob", "Alice"))
print("undirected: {Alice, Bob} == {Bob, Alice}?", friends == frozenset({"Bob", "Alice"}))
loop = frozenset({"v", "v"})
print("self-loop {v, v} =", set(loop), "→ a set with", len(loop), "element")


def as_undirected(edges) -> set:
    """Turn a list of pairs into a set of undirected edges {u, v} (frozensets); duplicates collapse."""
    return {frozenset(edge) for edge in edges}


ppi_pairs = [("TP53", "MDM2"), ("MDM2", "TP53"), ("TP53", "EP300")]    # the same binding listed twice
print("\nas directed pairs:", len(set(ppi_pairs)), "edges | as undirected edges:", len(as_undirected(ppi_pairs)))
print("is the inflammation network a subset of V × V?",
      set(NET_EDGES) <= set(itertools.product(NET_NODES, NET_NODES)))

🔍 **What just happened?** The ordered pair (Alice, Bob) differs from (Bob, Alice), but the set {Alice, Bob} equals
{Bob, Alice}: Python's `frozenset` (an immutable set) is the natural type for an undirected edge. A self-loop {v, v}
is the one-element set {v}. If a protein–protein interaction database lists the binding TP53–MDM2 in both directions,
the undirected view correctly counts it once (2 edges instead of 3). Finally, every edge of the inflammation network is
an element of $V\times V$, so it is a legitimate directed graph — a relation on the set of genes (Module 04).

**Which kind of graph for which biology?** Physical binding is mutual, so **protein–protein interaction (PPI)**
networks are undirected. Regulation has a direction — a transcription factor acts on its target, not vice versa — so
**gene-regulatory networks** are directed, as are signalling cascades and food webs. The next cell draws both kinds.

In [ ]:
# ── 1.2 Drawing directed and undirected graphs (a PPI fragment and the inflammation network) ──
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1, 2.6]})
draw_graph([("Alice", "Bob")], directed=True, pos={"Alice": (0, 0), "Bob": (1, 0)}, ax=axes[0], font_size=8,
           title="directed: Alice follows Bob")
draw_graph([("Alice", "Bob")], directed=False, pos={"Alice": (0, 0), "Bob": (1, 0)}, ax=axes[1], font_size=8,
           title="undirected: Alice and Bob are friends")
draw_graph(NET_EDGES, directed=True, pos=NET_POS, ax=axes[2], font_size=7,
           title="inflammation network (illustrative): regulator → target")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The first two panels show the same two people with an arrow (one-way relationship) and with a
plain line (mutual relationship). The third panel draws the 15 directed edges of the inflammation network at the
hand-chosen positions: NF-κB (NFKB1) in the centre with arrows to five targets, two pairs of arrows that run in both
directions between NFKB1 and TNF and between NFKB1 and IL1B (drawn as two slightly curved arrows), the IL6 → STAT3 →
acute-phase branch, and on the right a separate interferon module around STAT1. Even this tiny network already shows
the structural features this module names: hubs, feedback loops, cascades and separate components.

✏️ **Your turn.** (a) Add a self-loop `("STAT3", "STAT3")` (STAT3 does induce its own gene) to a copy of `NET_EDGES` and
redraw. How does `draw_graph` show it? (b) Is "is the parent of" (among people) directed or undirected? And "is a
sibling of"? Can either have self-loops?

---
## 2 · Directed graphs as skeletons of Bayesian networks

> 📍 **Where we are.** Directed graphs are also the backbone of the probabilistic models of the second half of this
> course. Some people use the phrase **Bayesian network** for a directed graph endowed with a probability distribution
> (Module 20); here we only learn the graph vocabulary.

In a directed graphical model, each vertex corresponds to a random variable, and if $(X, Y)\in E$ there is an arrow
pointing from $X$ to $Y$. Three examples:

* a graph over three variables $a, b, c$ with edges $a\to b$, $a\to c$, $b\to c$ — it represents the factorisation of
  a joint distribution as $p(a, b, c) = p(a)\,p(b\mid a)\,p(c\mid a, b)$ (*in words:* first draw $a$, then $b$ given its
  parent $a$, then $c$ given its parents $a$ and $b$ — Module 17 explains the notation $p(\cdot\mid\cdot)$);
* the graph with $V = \{X, Y, Z\}$ and $E = \{(Y, X), (Y, Z)\}$ — a common cause $Y$ of $X$ and $Z$, like a
  transcription factor and two of its target genes;
* overweight → heart disease ← smoking → cough.

**Vocabulary.** If an arrow connects $X$ and $Y$ (in either direction), $X$ and $Y$ are **adjacent**. If there is an
arrow from $X$ to $Y$, then $X$ is a **parent** of $Y$ and $Y$ is a **child** of $X$. A **directed path** between two
variables is a set of arrows all pointing in the same direction linking one variable to the other:
$X\to\cdots\to Y$.

In [ ]:
# ── 2.1 Parents, children, adjacency and directed paths in three small directed graphs ──
def parents(v, edges) -> set:
    """Pa(v) = {u | (u, v) ∈ E}: the nodes with an arrow into v."""
    return {u for (u, w) in edges if w == v}


def children(v, edges) -> set:
    """Ch(v) = {w | (v, w) ∈ E}: the nodes v points to."""
    return {w for (u, w) in edges if u == v}


def has_directed_path(source, target, edges) -> bool:
    """Is there a directed path source → … → target with at least one edge? (follow arrows from source)"""
    reached, frontier = set(), [source]
    while frontier:
        u = frontier.pop()
        for w in children(u, edges) - reached:
            reached.add(w)
            frontier.append(w)
    return target in reached


abc = [("a", "b"), ("a", "c"), ("b", "c")]
common_cause = [("Y", "X"), ("Y", "Z")]
health = [("overweight", "heart disease"), ("smoking", "heart disease"), ("smoking", "cough")]

print("parents of c:", sorted(parents("c", abc)), "| children of Y:", sorted(children("Y", common_cause)))
print("parents of heart disease:", sorted(parents("heart disease", health)),
      "| children of smoking:", sorted(children("smoking", health)))
adjacent = lambda u, v, E: (u, v) in E or (v, u) in E
print("overweight and smoking adjacent?", adjacent("overweight", "smoking", health),
      "| heart disease and smoking adjacent?", adjacent("heart disease", "smoking", health))
print("directed path smoking → cough?", has_directed_path("smoking", "cough", health),
      "| overweight → cough?", has_directed_path("overweight", "cough", health))
print("directed path IL6 → HAMP in the inflammation network?", has_directed_path("IL6", "HAMP", NET_EDGES),
      "| HAMP → IL6?", has_directed_path("HAMP", "IL6", NET_EDGES))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))
draw_graph(abc, directed=True, pos={"a": (0, 1), "b": (1, 0.5), "c": (0.4, 0)}, ax=axes[0], title="a → b, a → c, b → c")
draw_graph(common_cause, directed=True, pos={"Y": (0.5, 1), "X": (0, 0), "Z": (1, 0)}, ax=axes[1],
           title="E = {(Y, X), (Y, Z)}")
draw_graph(health, directed=True, font_size=7, ax=axes[2], title="smoking and heart disease",
           pos={"overweight": (0, 1), "smoking": (1.6, 1), "heart disease": (0.8, 0), "cough": (2.4, 0)})
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the three-variable graph, $c$ has the parents $a$ and $b$; in the common-cause graph $Y$
has the children $X$ and $Z$. Heart disease has two parents (overweight and smoking), smoking two children (heart
disease and cough). Overweight and smoking are **not adjacent** — no arrow joins them — while heart disease and smoking
are. There is a directed path smoking → cough but none from overweight to cough. In the inflammation network the
directed path IL6 → STAT3 → HAMP exists, but not the reverse: direction matters for "who can influence whom". In a
Bayesian network, the parents of a node are exactly the variables its probability table is conditioned on (Module 20).

✏️ **Your turn.** List the parents and children of NFKB1 in the inflammation network with `parents` and `children`.
Which genes are both a parent and a child of NFKB1, and what biological feature does that reveal?

---
## 3 · Paths, labels and weights

> 📍 **Where we are.** The most important question about a graph is often *how do you get from here to there?* Train
> and flight maps, cell interactions, signalling cascades and social networks are all graphs in which we follow edges.

A **path** is a sequence of nodes $\langle v_1, v_2, \dots, v_k\rangle$ ($k\ge 1$) such that every consecutive pair is an
edge:

$$\langle v_i, v_{i+1}\rangle\in E \ \text{ (if } G \text{ is directed)} \quad\text{or}\quad \{v_i, v_{i+1}\}\in E \ \text{ (if undirected)}, \qquad i = 1, \dots, k-1 .$$

*In words:* a path is a walk along edges — a tuple of nodes in which each node is joined by an edge to the next one; it
is a path **from** $v_1$ **to** $v_k$. (In this course a path may revisit a node. Many textbooks call such a sequence a
*walk* and keep the word *path* for walks that never repeat a node; Section 11 counts walks in exactly our sense.)

**Length of a path.** In this course the **length** of the path $\langle v_1, \dots, v_k\rangle$ is its number of
**edges**, $k - 1$; a single node $\langle v_1\rangle$ is a path of length 0.

⚠️ **Common confusion.** Some texts call the same path "of length $k$" (counting nodes). Both conventions exist; always
check which one is used. Counting edges is the standard in graph theory and matches adjacency matrices (Section 11),
where $A^\ell$ counts walks of $\ell$ edges.

**Example: course schedules.** Three students take courses in order — Student 1: Math → Physics → Chemistry → Art;
Student 2: Math → Physics → Art; Student 3: Math → History → Economics → Art. Treat each course as a node and connect two
courses if at least one student takes them consecutively. Each student's schedule is then a path in the graph.

In [ ]:
# ── 3.1 The course-schedule graph: edges from consecutive courses; every student's schedule is a path ──
schedules = {"Student 1": ("Math", "Physics", "Chemistry", "Art"),
             "Student 2": ("Math", "Physics", "Art"),
             "Student 3": ("Math", "History", "Economics", "Art")}
course_edges = sorted({(s[i], s[i + 1]) for s in schedules.values() for i in range(len(s) - 1)})
print("E =", course_edges, "→", len(course_edges), "edges")


def is_path(nodes: tuple, edges, directed: bool = True) -> bool:
    """⟨v1, …, vk⟩ is a path iff every consecutive pair is an edge (k ≥ 1)."""
    E = set(edges) if directed else as_undirected(edges)
    pair = (lambda u, v: (u, v)) if directed else (lambda u, v: frozenset((u, v)))
    return len(nodes) >= 1 and all(pair(nodes[i], nodes[i + 1]) in E for i in range(len(nodes) - 1))


def all_paths(start, end, edges, prefix=()) -> list:
    """Every directed path from start to end that never repeats a node (depth-first enumeration)."""
    prefix = prefix + (start,)
    if start == end:
        return [prefix]
    return [p for (u, w) in edges if u == start and w not in prefix for p in all_paths(w, end, edges, prefix)]


for student, s in schedules.items():
    print(f"{student}: {' → '.join(s)} is a path? {is_path(s, course_edges)} | length {len(s) - 1} edges")
print("Math → Chemistry → Art is a path?", is_path(("Math", "Chemistry", "Art"), course_edges))
paths_math_art = all_paths("Math", "Art", course_edges)
print("all paths Math → Art:", [" → ".join(p) for p in paths_math_art])

course_pos = {"Math": (0, 0), "Physics": (1.2, 0.8), "Chemistry": (2.4, 1.3), "History": (1.2, -0.8),
              "Economics": (2.4, -1.3), "Art": (3.6, 0)}
draw_graph(course_edges, directed=True, pos=course_pos, font_size=8, title="course-schedule graph")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The three schedules produce 7 distinct edges (Math → Physics appears in two schedules but is
one edge, because $E$ is a set). Each schedule is a path — of length 3, 2 and 3 edges — while
⟨Math, Chemistry, Art⟩ is not, because no student goes from Math straight to Chemistry. The depth-first enumeration
finds exactly the three paths from Math to Art; here they are the three students' schedules, but in general a graph
contains paths that nobody took (in a larger cohort, combining pieces of different schedules).

### 3.2 Labelled and weighted edges

We can also **label** or **weight** edges. In a route graph (for navigation or delivery), each edge might carry a street
name — a string — and in other graphs a number such as a distance, a time or a cost, which we call a **weight**.
Formally, instead of assigning labels by hand we define **functions** (Module 03) on the edge set:

$$\ell: E\to\Sigma^* \quad\text{(edge labels: strings)}, \qquad w: E\to\mathbb R \quad\text{(edge weights: real numbers)}.$$

*In words:* a labelling assigns to every edge one string over an alphabet Σ, a weighting assigns to every edge one real
number. Paths inherit them: a **path label** is the concatenation of its edge labels (Module 04), e.g.
"Main St → Broadway → 5th Ave", and a **path weight** is the sum of its edge weights, e.g. the total distance or travel
time.

| Function | Purpose | Output type |
|---|---|---|
| $\ell$ | edge label | a string (from $\Sigma^*$) |
| $w$ | edge weight | a real number (from $\mathbb R$) |

The next cell uses a small part of the US Interstate system between Los Angeles and Orlando (Florida), each road
labelled by its name; we add rounded, illustrative distances in miles as weights.

In [ ]:
# ── 3.2 A route graph with a label function ℓ: E → Σ* and a weight function w: E → ℝ (distances illustrative) ──
CITY = {"LA": "Los Angeles", "LC": "Lake City", "JAX": "Jacksonville", "TPA": "Tampa", "DB": "Daytona Beach",
        "ORL": "Orlando"}
road_label = {("LA", "LC"): "I10(west)", ("LC", "JAX"): "I10(east)", ("LC", "TPA"): "I75",
              ("JAX", "DB"): "I95", ("TPA", "ORL"): "I4(west)", ("DB", "ORL"): "I4(east)"}       # ℓ
road_miles = {("LA", "LC"): 2350, ("LC", "JAX"): 60, ("LC", "TPA"): 165,
              ("JAX", "DB"): 90, ("TPA", "ORL"): 85, ("DB", "ORL"): 55}                          # w (rounded)


def edge_value(table: dict, u, v):
    """Look up an undirected edge {u, v} in a table keyed by one orientation."""
    return table[(u, v)] if (u, v) in table else table[(v, u)]


def path_label(path: tuple, labels: dict) -> str:
    """Concatenation of the edge labels along the path."""
    return " · ".join(edge_value(labels, path[i], path[i + 1]) for i in range(len(path) - 1))


def path_weight(path: tuple, weights: dict) -> float:
    """Sum of the edge weights along the path."""
    return sum(edge_value(weights, path[i], path[i + 1]) for i in range(len(path) - 1))


for route in [("LA", "LC", "TPA", "ORL"), ("LA", "LC", "JAX", "DB", "ORL")]:
    print(" → ".join(route), f"| label: {path_label(route, road_label)} | weight: {path_weight(route, road_miles):,} miles")
neighbours_LC = sorted({v for e in road_label for v in e if "LC" in e and v != "LC"})
print("neighbourhood of Lake City:", [CITY[c] for c in neighbours_LC], "| degree", len(neighbours_LC))

route_pos = {"LA": (0, 1), "LC": (3, 1), "JAX": (4.4, 1), "TPA": (3, 0), "DB": (4.4, 0), "ORL": (3.7, -0.8)}
fig, ax = plt.subplots(figsize=(6.6, 3.2))
draw_graph(list(road_label), pos=route_pos, edge_labels=road_label, font_size=8, ax=ax,
           title="route graph: ℓ(edge) = road name\n" + ", ".join(f"{k} = {v}" for k, v in CITY.items()))
plt.tight_layout(); plt.show()

🔍 **What just happened?** The two routes from Los Angeles (LA) to Orlando (ORL) have different path labels — the
concatenations "I10(west) · I75 · I4(west)" and "I10(west) · I10(east) · I95 · I4(east)" are the driving directions —
and different path weights, 2,600 and 2,555 (illustrative) miles: the route with *more* edges is shorter, a reminder
that path length (edges) and path weight (distance) are different things. Lake City has the neighbourhood
{Los Angeles, Tampa, Jacksonville}, so its degree is 3 (Section 4). On the drawing, the dark red text on each edge is
the value of the function ℓ.

### 3.3 Genomes as graphs: path labels are haplotypes

In computational biology we also represent **genomes as graphs**: a directed graph in which genetic variation appears
as alternative **paths** from start to finish. Each edge carries a string (a piece of DNA), and the label of a path —
the concatenation — is one possible sequence (a haplotype). Such **variation graphs** (or pangenome graphs) store a
reference genome together with known variants. Below, a SNP (T or C) and a two-base deletion (TT present or absent) give
four haplotypes.

In [ ]:
# ── 3.3 A tiny genome (variation) graph: edge labels are DNA strings, path labels are haplotypes ──
segment = {"s1": "ACG", "s2": "T", "s3": "C", "s4": "GA", "s5": "TT", "s6": "CAT"}       # DNA carried by each node
genome_edges = [("s1", "s2"), ("s1", "s3"), ("s2", "s4"), ("s3", "s4"), ("s4", "s5"), ("s5", "s6"), ("s4", "s6")]
genome_label = {(u, v): segment[v] for (u, v) in genome_edges}           # ℓ(u → v) = the DNA of the node entered

haplotypes = []
for p in all_paths("s1", "s6", genome_edges):
    sequence = segment["s1"] + "".join(genome_label[(p[i], p[i + 1])] for i in range(len(p) - 1))
    haplotypes.append(sequence)
    print(" → ".join(p), "| path label:", sequence, f"({len(sequence)} bp)")
print("number of haplotypes = number of paths from s1 to s6 =", len(haplotypes))

genome_pos = {"s1": (0, 0), "s2": (1, 0.6), "s3": (1, -0.6), "s4": (2, 0), "s5": (3, 0.6), "s6": (4, 0)}
draw_graph(genome_edges, directed=True, pos=genome_pos, edge_labels=genome_label,
           title="variation graph: a SNP (T/C) and a deletion (TT / no TT)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Starting from s1 (ACG), every path to s6 spells one haplotype: ACGTGATTCAT, ACGTGACAT,
ACGCGATTCAT and ACGCGACAT — 11 or 9 bases long. Two independent binary choices (SNP allele, deletion or not) give
$2\times 2 = 4$ paths — the multiplication principle of Module 04 in graph form. Real pangenome graphs encode millions of
variants this way, and aligning a sequencing read to the graph means finding the path whose label matches it best.

✏️ **Your turn.** (a) Add a second SNP by splitting s6 into "CA" + ("T" or "G"). How many haplotypes do you expect? (b)
In 3.2, add an illustrative edge LA–TPA with label "I10·I75 (bypass)" and weight 2,450; which route is now the
shortest?

### 📘 Lecture notes — graphs, directed graphs and paths

**Graph.** A mathematical structure that models pairwise relationships: $G = \langle V, E\rangle$, with $V$ a
(non-empty) set of nodes or vertices and $E$ a set of edges, $E\subseteq V\times V$ if $G$ is directed and
$E\subseteq\{\{u,v\}: u, v\in V\}$ if $G$ is undirected (does the order of the nodes in an edge matter?). A graph is also
called a network. Directed edge = ordered pair, "from $u$ to $v$", $(u, v)\ne(v, u)$; undirected edge = unordered pair,
"between $u$ and $v$", $\{u, v\} = \{v, u\}$; $u = v$ is allowed and gives a **self-loop**.

**Examples.** Twitter follows (A → B, directed) vs Facebook friendships (A — B, undirected); train/flight maps, cell
interactions, social networks. Drawing: nodes as circles, edges as lines, arrows for directed edges.

**Directed graphs and Bayesian networks.** A Bayesian network is a directed graph endowed with a probability
distribution; e.g. $V = \{X, Y, Z\}$, $E = \{(Y, X), (Y, Z)\}$; overweight → heart disease ← smoking → cough. Adjacent
(an arrow in either direction), parent/child (arrow from parent to child), directed path $X\to\cdots\to Y$.

**Paths.** $\langle v_1, \dots, v_k\rangle$, $k\ge 1$, with $\langle v_i, v_{i+1}\rangle\in E$ (directed) or
$\{v_i, v_{i+1}\}\in E$ (undirected): a path from $v_1$ to $v_k$. Its length is the number of edges, $k-1$ (some
sources say $k$ — state your convention).

**Labels and weights.** Functions $\ell: E\to\Sigma^*$ and $w: E\to\mathbb R$; path label = concatenation of edge
labels, path weight = sum of edge weights (route graph: "I10(west) · I75 · I4(west)", 2,600 illustrative miles). In
genome graphs, path labels are haplotypes.

> 🎓 **Exam tip.** Write graphs explicitly as $\langle V, E\rangle$ with set notation, use ( ) for directed and { } for
> undirected edges, and always say whether a path length counts edges or nodes.

---
## 4 · Neighbourhoods and degrees

> 📍 **Where we are.** Paths tell us how to travel; the simplest local question is *who is directly connected to
> whom?* The answer is a node's **neighbourhood**, and its size — the **degree** — is the most used number in network
> biology (a protein's number of interaction partners, a person's number of friends).

Let $G = \langle V, E\rangle$ be an undirected graph and $v\in V$. The **neighbourhood** of $v$ is the set of all
neighbours of $v$:

$$N_G(v) = \{u\in V \mid \{u, v\}\in E\}.$$

*In words:* $u$ is a neighbour of $v$ when there is an edge connecting $u$ and $v$; $N_G(v)$ collects all of them.
(With a self-loop $\{v, v\}\in E$, $v$ is its own neighbour.) In the route graph, the neighbourhood of Lake City is
{Los Angeles, Tampa, Jacksonville}; in a social network, the neighbourhood of a person is the set of their friends.

**Indicator functions.** Given a set $X$ and a subset $A\subseteq X$, the indicator function of $A$ is
$\mathbb 1_A: X\to\{0, 1\}$ with $\mathbb 1_A(x) = 1$ if $x\in A$ and $0$ if $x\notin A$. Applied to statements,
$\mathbb 1:\{\text{true}, \text{false}\}\to\{1, 0\}$ turns "true" into 1 and "false" into 0 — a numerical form of a
condition (Module 03).

**Degree.** The degree of $v$ is the size of its neighbourhood, plus 1 if $v$ is in its own neighbourhood:

$$\operatorname{degree}_G(v) := |N_G(v)| + \mathbb 1\big(v\in N_G(v)\big).$$

*In words:* count the neighbours; if there is a self-loop, add one more — so that a loop contributes **2** to the degree
of its single endpoint. This agrees with the textbook rule: *the degree of a vertex in an undirected graph is the number
of edges incident with it, except that a loop at a vertex contributes twice*, written $\deg(v)$. Lake City has degree 3
and Los Angeles degree 1; in a social network the degree is a person's popularity — the number of friends they have.

⚠️ **Common confusion.** "The degree is the number of edges incident to $v$" is only correct if a loop is counted as
incident **twice** (it touches $v$ at both of its ends). Counting the loop once gives a different number and breaks the
handshake lemma below.

In [ ]:
# ── 4.1 Neighbourhood, indicator function and degree; the two examples without and with a self-loop ──
def indicator(condition: bool) -> int:
    """𝟙(condition): 1 if the condition is true, 0 otherwise."""
    return 1 if condition else 0


def neighbourhood(v, edges) -> set:
    """N_G(v) = {u | {u, v} ∈ E} for an undirected graph given as a list of pairs (a loop puts v in N(v))."""
    return {b for (a, b) in edges if a == v} | {a for (a, b) in edges if b == v}


def degree(v, edges) -> int:
    """degree_G(v) := |N_G(v)| + 𝟙(v ∈ N_G(v)) — a self-loop counts twice."""
    N = neighbourhood(v, edges)
    return len(N) + indicator(v in N)


def edge_ends_at(v, edges) -> int:
    """Number of edge ends at v: an edge {u, v} with u ≠ v contributes 1, a loop {v, v} contributes 2."""
    return sum((a == v) + (b == v) for (a, b) in edges)


for name, E in [("E = {{v,a}, {v,b}}", [("v", "a"), ("v", "b")]),
                ("E = {{v,a}, {v,b}, {v,v}}", [("v", "a"), ("v", "b"), ("v", "v")])]:
    N = neighbourhood("v", E)
    print(f"{name:27s}: N(v) = {sorted(N)}, 1(v ∈ N(v)) = {indicator('v' in N)}, degree = {len(N)} + "
          f"{indicator('v' in N)} = {degree('v', E)} | edge ends at v: {edge_ends_at('v', E)} | "
          f"edges merely touching v: {sum('v' in e for e in E)}")

🔍 **What just happened?** Without a loop, $N_G(v) = \{a, b\}$, $v\notin N_G(v)$, so the indicator is 0 and the degree is
$2 + 0 = 2$. With the loop $\{v, v\}$, $N_G(v) = \{a, b, v\}$, the indicator is 1 and the degree is $3 + 1 = 4$.
Counting edge *ends* gives the same numbers (2 and 4), which is the "loops count twice" rule. Counting the edges that
merely *touch* $v$ gives 3 in the second case — the inconsistent "alternative definition" that the warning above is
about.

### 4.2 The handshake lemma

**Theorem (handshake lemma).** For every undirected graph,

$$\sum_{v\in V}\deg(v) = 2\,|E| .$$

*In words:* adding up all degrees counts every edge exactly twice. *Proof.* Each edge $\{u, v\}$ with $u\ne v$ adds 1 to
$\deg(u)$ and 1 to $\deg(v)$; each loop $\{v, v\}$ adds 2 to $\deg(v)$. Either way, each edge adds exactly 2 to the
sum. ∎ (The name: if everyone at a party counts the hands they shook, the total is twice the number of handshakes.)

**Corollary.** The number of nodes with odd degree is even — because the odd degrees must add up to an even number.

**Exercise (textbook).** What are the degrees and neighbourhoods of the vertices $a, \dots, g$ in the graph drawn below?
Work it out from the picture before reading the printed answer.

In [ ]:
# ── 4.2 The degree exercise a…g: neighbourhoods, degrees and the handshake lemma ──
rosen_nodes = list("abcdefg")
rosen_edges = [("a", "b"), ("a", "f"), ("b", "f"), ("b", "c"), ("b", "e"), ("c", "f"), ("c", "e"), ("c", "d"), ("f", "e")]
rosen_pos = {"a": (0, 0), "b": (1, 1), "c": (2, 1), "d": (3, 1), "f": (1, 0), "e": (2, 0), "g": (3, 0)}
rows = []
for v in rosen_nodes:
    N = neighbourhood(v, rosen_edges)
    rows.append({"v": v, "N(v)": "{" + ", ".join(sorted(N)) + "}" if N else "∅", "deg(v)": degree(v, rosen_edges)})
print(pd.DataFrame(rows).to_string(index=False))
total = sum(degree(v, rosen_edges) for v in rosen_nodes)
print(f"Σ deg = {total} = 2|E| = 2 · {len(rosen_edges)}; odd-degree nodes: "
      f"{[v for v in rosen_nodes if degree(v, rosen_edges) % 2]} (an even number of them)")

draw_graph(rosen_edges, nodes=rosen_nodes, pos=rosen_pos, title="the degree exercise (g is isolated)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The table reproduces the textbook answer: $\deg(a) = 2$, $\deg(b) = \deg(c) = \deg(f) = 4$,
$\deg(d) = 1$, $\deg(e) = 3$ and $\deg(g) = 0$, with $N(a) = \{b, f\}$, $N(b) = \{a, c, e, f\}$, $N(c) = \{b, d, e, f\}$,
$N(d) = \{c\}$, $N(e) = \{b, c, f\}$, $N(f) = \{a, b, c, e\}$ and $N(g) = \varnothing$. The degrees add up to
$18 = 2\cdot 9$ edges, and exactly two nodes ($d$ and $e$) have odd degree. Node $g$ has degree 0: an **isolated** node
belongs to the graph but touches no edge (we had to pass it in `nodes=`, because it appears in no edge).

### 4.3 A second exercise — with parallel edges and a loop

The next textbook figure is a **multigraph**: several parallel edges may join the same two vertices. (In this course we
work with simple graphs — at most one edge between two vertices; multigraphs will not be examined. The exercise is
useful because it shows the degree convention: a self-loop counts 2 toward the degree of its single endpoint, and
adds 2 to the total sum of degrees.) Its answer: $\deg(a) = 4$, $\deg(b) = \deg(e) = 6$, $\deg(c) = 1$, $\deg(d) = 5$,
with $N(a) = \{b, d, e\}$, $N(b) = \{a, b, c, d, e\}$, $N(c) = \{b\}$, $N(d) = \{a, b, e\}$, $N(e) = \{a, b, d\}$.

In [ ]:
# ── 4.3 The multigraph exercise: count edge ends (loops twice); the neighbourhood formula needs simple graphs ──
from matplotlib.patches import FancyArrowPatch


def draw_multigraph(edges, pos: dict, ax, title: str = ""):
    """Draw an undirected multigraph: parallel edges as separate arcs, a self-loop as a small ring."""
    copies = Counter(frozenset(e) for e in edges)
    for pair, m in copies.items():
        ends = sorted(pair)
        if len(ends) == 1:                                           # a self-loop
            x, y = pos[ends[0]]
            ax.add_patch(plt.Circle((x + 0.12, y + 0.2), 0.17, fill=False, color="0.35", lw=1.3))
            continue
        u, v = ends
        for k in range(m):                                           # spread the m parallel copies
            rad = 0.0 if m == 1 else (k - (m - 1) / 2) * 0.45
            ax.add_patch(FancyArrowPatch(pos[u], pos[v], arrowstyle="-", connectionstyle=f"arc3,rad={rad}",
                                         color="0.35", lw=1.3, shrinkA=9, shrinkB=9))
    for v, (x, y) in pos.items():
        ax.scatter([x], [y], s=330, color="#cfe2f3", edgecolors="0.2", zorder=3)
        ax.text(x, y, v, ha="center", va="center", zorder=4)
    ax.set_aspect("equal"); ax.axis("off"); ax.margins(0.3); ax.set_title(title)


multi_edges = [("a", "b"), ("b", "c"), ("b", "b"), ("a", "d"), ("b", "e"), ("b", "d"),
               ("a", "e"), ("a", "e"), ("e", "d"), ("e", "d"), ("e", "d")]
multi_pos = {"a": (0, 1), "b": (1, 1), "c": (2, 1), "e": (0, 0), "d": (1, 0)}
rows = [{"v": v, "N(v)": "{" + ", ".join(sorted(neighbourhood(v, multi_edges))) + "}",
         "deg(v) = edge ends": edge_ends_at(v, multi_edges), "|N(v)| + 1(v ∈ N(v))": degree(v, multi_edges)}
        for v in "abcde"]
print(pd.DataFrame(rows).to_string(index=False))
print("Σ deg =", sum(edge_ends_at(v, multi_edges) for v in "abcde"), "= 2|E| = 2 ·", len(multi_edges))

fig, ax = plt.subplots(figsize=(4.6, 3.4))
draw_multigraph(multi_edges, multi_pos, ax, title="multigraph: 2 parallel a–e edges,\n3 parallel e–d edges, a loop at b")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Counting edge ends reproduces the answer: 4, 6, 1, 5, 6, summing to $22 = 2\cdot 11$ edges —
the handshake lemma holds for multigraphs too, with the loop at $b$ contributing 2. The neighbourhoods are the printed
sets. But the formula $|N(v)| + \mathbb 1(v\in N(v))$ gives 3 for $a$, 3 for $d$ and 3 for $e$: a neighbourhood is a
*set*, so parallel edges put a neighbour into it only once. The neighbourhood definition of degree is designed for
**simple graphs** (where it agrees with counting edge ends); for multigraphs, count edge ends. (For $b$ and $c$ the
formula happens to give the right 6 and 1, because they have no parallel edges.)

### 4.4 Question: neighbourhood and degree

For the graph below: What are the neighbours of node C? Which nodes have degree one? Which node has the highest degree?
Which nodes are in the neighbourhoods of both B and E? Answer on paper first.

In [ ]:
# ── 4.4 Answering the neighbourhood questions with set operations ──
q_nodes = list("ABCDEFGH")
q_edges = [("A", "B"), ("B", "C"), ("B", "E"), ("C", "E"), ("D", "G"), ("E", "F"), ("G", "H")]
q_pos = {"A": (0, 0.5), "B": (1, 1), "C": (1, 0), "D": (2, 1), "E": (2, 0), "F": (3, 1), "G": (3, 0), "H": (4, 0.5)}
deg_q = {v: degree(v, q_edges) for v in q_nodes}
print("N(C) =", sorted(neighbourhood("C", q_edges)))
print("degree 1:", [v for v in q_nodes if deg_q[v] == 1], "— their only neighbours:",
      [sorted(neighbourhood(v, q_edges))[0] for v in q_nodes if deg_q[v] == 1])
top = max(deg_q.values())
print("highest degree", top, ":", [v for v in q_nodes if deg_q[v] == top])
print("N(B) =", sorted(neighbourhood("B", q_edges)), "| N(E) =", sorted(neighbourhood("E", q_edges)),
      "| N(B) ∩ N(E) =", sorted(neighbourhood("B", q_edges) & neighbourhood("E", q_edges)))
draw_graph(q_edges, nodes=q_nodes, pos=q_pos, title="neighbourhood and degree question")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Node C has two neighbours, B and E. The nodes of degree one are A, D, F and H (their solitary
neighbours are B, G, E and G). B and E both have degree 3 and are tied as the nodes of highest degree. Their
neighbourhoods are {A, C, E} and {B, C, F}; the intersection — a set operation from Module 03 — is {C}, the only node
adjacent to both. In a protein network, "common neighbours" of two proteins is a classic hint that they work in the
same complex or pathway.

### 🌉 Bridge — hubs in protein-interaction networks

In protein–protein interaction networks the degree of a protein is its number of binding partners. Degrees are very
unequal: most proteins have few partners and a few **hubs** have very many — the tumour suppressor p53 (TP53) is a famous
example. In yeast, Jeong *et al.* (2001) found that highly connected proteins are more often essential for survival
(the "centrality–lethality" observation). A simple growth model explains such unequal degrees: if new nodes prefer to
attach to nodes that already have many links (**preferential attachment**, Barabási & Albert 1999), a few early nodes
become hubs. The next cell computes degrees in a small, typed-in set of well-known p53 interactions and compares a
SIMULATED preferential-attachment network with a SIMULATED random network having the same number of nodes and edges.

In [ ]:
# ── 4.5 Degrees in a small PPI network (typed in) and in two SIMULATED networks with 300 nodes ──
ppi_edges = [("TP53", "MDM2"), ("MDM2", "MDM4"), ("TP53", "EP300"), ("TP53", "ATM"), ("ATM", "CHEK2"),
             ("TP53", "CHEK2"), ("TP53", "BRCA1"), ("BRCA1", "BARD1")]          # illustrative subset of known bindings
ppi_nodes = sorted({p for e in ppi_edges for p in e})
ppi_degree = {p: degree(p, ppi_edges) for p in ppi_nodes}
print("PPI degrees:", dict(sorted(ppi_degree.items(), key=lambda kv: (-kv[1], kv[0]))),
      "| Σ =", sum(ppi_degree.values()), "= 2|E|")


def preferential_attachment_edges(n: int, m: int, seed: int) -> list:
    """SIMULATED network: start from a triangle; each new node links to m distinct nodes chosen with
    probability proportional to their current degree (every edge end is a 'ticket' in the pool)."""
    rng_pa = np.random.default_rng(seed)
    edges, pool = [(0, 1), (1, 2), (0, 2)], [0, 1, 1, 2, 0, 2]
    for new in range(3, n):
        targets = set()
        while len(targets) < m:
            targets.add(int(pool[rng_pa.integers(len(pool))]))
        for t in sorted(targets):
            edges.append((new, t))
            pool += [new, t]
    return edges


def random_edges(n: int, n_edges: int, seed: int) -> list:
    """SIMULATED random graph: n_edges distinct pairs drawn uniformly from all n(n−1)/2 pairs."""
    rng_er = np.random.default_rng(seed)
    all_pairs = list(itertools.combinations(range(n), 2))
    return [all_pairs[i] for i in rng_er.choice(len(all_pairs), size=n_edges, replace=False)]


pa_edges = preferential_attachment_edges(300, 2, seed=SEED)
er_edges = random_edges(300, len(pa_edges), seed=SEED)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, (name, E) in zip(axes, [("preferential attachment", pa_edges), ("uniformly random", er_edges)]):
    degs = Counter(v for e in E for v in e)
    values = np.array([degs.get(v, 0) for v in range(300)])
    print(f"{name:24s}: |E| = {len(E)}, mean degree {values.mean():.2f}, max degree {values.max()}, "
          f"nodes with degree ≥ 15: {(values >= 15).sum():2d}, Σ deg = 2|E|? {values.sum() == 2 * len(E)}")
    ax.hist(values, bins=np.arange(0, values.max() + 2) - 0.5, color="tab:purple" if "pref" in name else "tab:gray")
    ax.set_yscale("log"); ax.set_xlabel("degree"); ax.set_ylabel("number of nodes (log)")
    ax.set_title(f"SIMULATED, {name}: max degree {values.max()}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the small PPI set, TP53 has degree 5 and every other protein degree 1 or 2 — a hub-and-spoke
pattern; the degrees sum to $16 = 2\cdot 8$. Both simulated networks have 300 nodes, 597 edges and therefore the same
mean degree $2\cdot 597/300 = 3.98$ (the handshake lemma again). But their degree distributions differ completely: the
uniformly random network has a maximum degree of 11 and no node with degree 15 or more, while preferential attachment
creates a long tail of hubs: 8 nodes with degree ≥ 15 and a maximum of 40, ten times the mean (note the logarithmic
count axis). Real interaction networks
look more like the left panel, which is why "hub" proteins are a central idea in network biology and why
graph-based machine learning must cope with very unequal neighbourhood sizes (BiologyToML
[Module 28 · graph neural networks](../../../../ml/tutorial/BiologyToML/notebooks/28_graph_neural_networks.ipynb)).

✏️ **Your turn.** Compute the degree of every gene in the inflammation network *ignoring directions* (use
`degree(g, NET_EDGES)`), and check the handshake lemma. Which gene is the hub?

---
## 5 · Directed degrees, regular graphs and complete graphs

> 📍 **Where we are.** In a regulatory network it matters whether a gene *regulates* many genes (a master regulator) or
> *is regulated by* many (an integrator of signals). Directed graphs split the neighbourhood in two.

For an edge $\langle u, v\rangle$ from $u$ to $v$ in a directed graph, $v$ is an **out-neighbour** of $u$ and $u$ is an
**in-neighbour** of $v$. For a node $v$:

$$N^{\text{in}}_G(v) = \{u \mid \langle u, v\rangle\in E\}, \qquad N^{\text{out}}_G(v) = \{u \mid \langle v, u\rangle\in E\}, \qquad N_G(v) = N^{\text{in}}_G(v)\cup N^{\text{out}}_G(v).$$

*In words:* the in-neighbourhood of $v$ contains the nodes with an arrow **into** $v$, the out-neighbourhood the nodes $v$
points **to**, and the (whole) neighbourhood is their union; all nodes of $N_G(v)$ are called **adjacent** to $v$. The
**in-degree** and **out-degree** are the sizes $|N^{\text{in}}_G(v)|$ and $|N^{\text{out}}_G(v)|$. For a flight network
(airports, directed flights), the out-neighbours of an airport are those with direct flights *from* it, and the
in-neighbours those with direct flights *to* it.

**Notation.** We often leave off the $G$ and write $N$, $N^{\text{in}}$, $N^{\text{out}}$, degree when the graph is
clear from context.

In [ ]:
# ── 5.1 In- and out-neighbours of a node u in a small flight network (illustrative) ──
def in_neighbours(v, edges) -> set:
    """N_in(v) = {u | (u, v) ∈ E}."""
    return {u for (u, w) in edges if w == v}


def out_neighbours(v, edges) -> set:
    """N_out(v) = {w | (v, w) ∈ E}."""
    return {w for (u, w) in edges if u == v}


flights = [("p", "u"), ("q", "u"), ("r", "u"), ("u", "r"), ("u", "s"), ("u", "t"), ("u", "w")]
flight_pos = {"u": (0, 0), "p": (-1, 0.5), "q": (-1, -0.5), "r": (0, 1), "s": (1, 0.6), "t": (1.1, 0), "w": (1, -0.6)}
N_in, N_out = in_neighbours("u", flights), out_neighbours("u", flights)
print("N_in(u) =", sorted(N_in), "| N_out(u) =", sorted(N_out), "| N(u) = N_in ∪ N_out =", sorted(N_in | N_out))
print("in-degree", len(N_in), "| out-degree", len(N_out), "| r is in both: there are flights both ways")
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, chosen, title in [(axes[0], N_in, "(a) in-neighbours of u (flights TO u)"),
                          (axes[1], N_out, "(b) out-neighbours of u (flights FROM u)")]:
    colours = {v: ("#6aa84f" if v in chosen else "#eeeeee") for v in flight_pos}
    colours["u"] = "#ffd966"
    draw_graph(flights, directed=True, pos=flight_pos, ax=ax, node_colors=colours, title=title)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Airport $u$ receives flights from $p$, $q$ and $r$ (in-degree 3) and sends flights to $r$, $s$,
$t$ and $w$ (out-degree 4). $r$ is in both sets — there are flights both ways — so the whole neighbourhood $N(u)$ has
6 nodes, not 7: the union counts $r$ once.

### 5.2 Example: the A–H directed graph

For the directed graph drawn below: What are the in-neighbours of C? Its out-neighbours? Which nodes are in both the
in-neighbourhood and the out-neighbourhood of E? Which nodes have in-degree zero? Out-degree zero?

In [ ]:
# ── 5.2 The A–H directed example: in/out-neighbourhoods and degrees ──
ah_edges = [("A", "B"), ("B", "C"), ("B", "E"), ("E", "B"), ("C", "D"), ("C", "E"), ("D", "G"),
            ("E", "F"), ("F", "E"), ("F", "H"), ("H", "G")]
ah_nodes = list("ABCDEFGH")
ah_pos = {"A": (0, 0.3), "B": (1, 1.1), "C": (1.2, -0.7), "D": (2.2, 1.3), "E": (2.6, -0.9), "F": (3.7, 1.3),
          "G": (3.6, -0.3), "H": (4.6, 0.6)}
fmt = lambda S: "{" + ", ".join(sorted(S)) + "}" if S else "∅"
table_ah = pd.DataFrame({"N_in": [fmt(in_neighbours(v, ah_edges)) for v in ah_nodes],
                         "N_out": [fmt(out_neighbours(v, ah_edges)) for v in ah_nodes],
                         "in-degree": [len(in_neighbours(v, ah_edges)) for v in ah_nodes],
                         "out-degree": [len(out_neighbours(v, ah_edges)) for v in ah_nodes]}, index=ah_nodes)
print(table_ah.to_string())
print("\nN_in(E) ∩ N_out(E) =", sorted(in_neighbours("E", ah_edges) & out_neighbours("E", ah_edges)))
print("in-degree 0:", [v for v in ah_nodes if not in_neighbours(v, ah_edges)],
      "| out-degree 0:", [v for v in ah_nodes if not out_neighbours(v, ah_edges)])
print("Σ in-degrees =", table_ah["in-degree"].sum(), "= Σ out-degrees =", table_ah["out-degree"].sum(), "= |E| =", len(ah_edges))
colours = {v: "#cfe2f3" for v in ah_nodes}
colours.update({"C": "#ffd966", "B": "#b6d7a8", "D": "#f4cccc", "E": "#f4cccc"})
draw_graph(ah_edges, directed=True, pos=ah_pos, node_colors=colours,
           title="C (yellow): in-neighbour B (green), out-neighbours D, E (red)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Node C has one in-neighbour, B, and two out-neighbours, D and E. Node E has three in-neighbours
(B, C and F) and two out-neighbours (B and F), so B and F are in both of E's neighbourhoods — E and B, and E and F, are
joined by arrows in both directions. A has no in-neighbours (in-degree 0: a "source", like a receptor that starts a
signal) and G has no out-neighbours (out-degree 0: a "sink"). The directed handshake lemma is simpler than the
undirected one: every edge has exactly one tail and one head, so $\sum_v\text{in-degree} = \sum_v\text{out-degree} = |E| = 11$.

### 5.3 Regulators, targets and feedback in the inflammation network

In [ ]:
# ── 5.3 In/out-degrees of the inflammation network; feedback loops and a feed-forward loop ──
deg_table = pd.DataFrame({"in-degree": [len(in_neighbours(g, NET_EDGES)) for g in NET_NODES],
                          "out-degree": [len(out_neighbours(g, NET_EDGES)) for g in NET_NODES]}, index=NET_NODES)
print(deg_table.sort_values(["out-degree", "in-degree"], ascending=False).T.to_string())
two_cycles = sorted({tuple(sorted((u, v))) for (u, v) in NET_EDGES if (v, u) in NET_EDGES})
print("\nfeedback loops of length 2 (u → v → u):", two_cycles)
ffl = [(a, b, c) for (a, b) in NET_EDGES for (b2, c) in NET_EDGES if b2 == b and (a, c) in NET_EDGES and a != c]
print("feed-forward loops a → b → c with a shortcut a → c:", ffl)
print("pure regulators (in-degree 0):", [g for g in NET_NODES if deg_table.loc[g, "in-degree"] == 0],
      "| pure targets (out-degree 0):", [g for g in NET_NODES if deg_table.loc[g, "out-degree"] == 0])

🔍 **What just happened?** NFKB1 is the hub regulator with out-degree 5 (IL1B, TNF, IL6, CXCL8, LYZ); STAT1 and STAT3
follow with 3 targets each. NFKB1 also has in-degree 2: TNF and IL-1β activate NF-κB, giving two **feedback loops** of
length 2, NFKB1 ⇄ TNF and NFKB1 ⇄ IL1B — positive feedback that can sustain inflammation (and that anti-TNF drugs
interrupt). STAT1 → IRF1 → ISG15 together with the shortcut STAT1 → ISG15 forms a **feed-forward loop**, one of the
recurring "network motifs" of transcriptional networks (Milo *et al.* 2002). STAT1 is the only pure regulator — nothing
in this small network regulates it (in reality interferon signalling does) — and seven genes are pure targets with
out-degree 0: the three acute-phase genes C3, HAMP and SAA1, the chemokine CXCL8, lysozyme LYZ and the
interferon-stimulated genes MX1 and ISG15.

### 5.4 Regular graphs

A graph is **regular** if every node has the same degree:

$$\forall v\in V:\ \operatorname{degree}_G(v) = k \quad\text{for some fixed } k .$$

*In words:* there is one number $k$ that is the degree of every node. More precisely, for an integer $d\ge 0$, a
**$d$-regular** graph is one in which every node has degree exactly $d$; a graph is regular if it is $d$-regular for
some $d$. The term usually refers to undirected graphs; for directed graphs, a regular graph requires all nodes to have
identical in-degrees and out-degrees — for example a **2-regular directed graph on 4 vertices**, in which every vertex
has in-degree 2 and out-degree 2.

### 5.5 Complete graphs and cliques

A **complete graph** (or **clique**) is an undirected graph $G = \langle V, E\rangle$ such that $\{u, v\}\in E$ for any
two distinct nodes $u, v\in V$: every two distinct nodes are directly connected, there are no missing connections, and
the graph is maximally connected. With $V = \{A, B, C\}$ the edge set is $E = \{\{A,B\}, \{A,C\}, \{B,C\}\}$. In set
notation, $E = \{\{u, v\} : u\ne v\in V\}$ (undirected) or $E = V\times V\setminus\{\langle v, v\rangle \mid v\in V\}$
(directed). The complete graph on $n$ nodes is written $K_n$.

Every node of $K_n$ is connected to the other $n-1$ nodes, so **$K_n$ is $(n-1)$-regular**, and by the handshake lemma

$$|E(K_n)| = \frac{1}{2}\sum_{v}\deg(v) = \frac{n(n-1)}{2} = \binom{n}{2}.$$

*In words:* a complete graph has one edge per pair of nodes; that is $n(n-1)/2$ edges, which grows like $n^2$
(Module 05: $\Theta(n^2)$).

The word **clique** is also used for a *subgraph* that is complete — a set of nodes in which every possible edge
actually exists. The graph with nodes {A, B, C, D} and edges {A,B}, {A,C}, {B,C}, {C,D} contains the 3-node clique
{A, B, C}.

In [ ]:
# ── 5.4 Which graphs are regular? Undirected examples and a 2-regular directed graph on 4 vertices ──
def regularity(nodes, edges, directed: bool = False):
    """Return d if the graph is d-regular — every node has degree d (undirected), or in-degree d and
    out-degree d (directed); otherwise None."""
    nodes = list(nodes)
    if directed:
        pairs = {(len(in_neighbours(v, edges)), len(out_neighbours(v, edges))) for v in nodes}
        if len(pairs) == 1:
            d_in, d_out = pairs.pop()
            return d_in if d_in == d_out else None
        return None
    degrees = {degree(v, edges) for v in nodes}
    return degrees.pop() if len(degrees) == 1 else None


K4_nodes = list("ABCD")
K4_edges = list(itertools.combinations(K4_nodes, 2))
cycle5 = [(i, (i + 1) % 5) for i in range(5)]
directed_2reg = [(i, (i + 1) % 4) for i in range(4)] + [(i, (i + 2) % 4) for i in range(4)]
print("K4 is", regularity(K4_nodes, K4_edges), "-regular | the 5-cycle is", regularity(range(5), cycle5), "-regular")
print("the degree exercise a…g is regular?", regularity(rosen_nodes, rosen_edges))
print("directed graph i → i+1 and i → i+2 (mod 4):", directed_2reg)
print("  (in, out)-degrees:", [(len(in_neighbours(v, directed_2reg)), len(out_neighbours(v, directed_2reg))) for v in range(4)],
      "→ d =", regularity(range(4), directed_2reg, directed=True))

🔍 **What just happened?** $K_4$ is 3-regular and the 5-cycle (a ring of five nodes) is 2-regular, while the degree
exercise graph is not regular (`None`): its degrees range from 0 to 4. The directed graph in which each of the four
vertices points to the next one and to the one after that has in-degree 2 and out-degree 2 everywhere — a 2-regular
directed graph on 4 vertices.

In [ ]:
# ── 5.5 Complete graphs K3, K4, K5, K6: (n−1)-regular with n(n−1)/2 edges; finding the clique {A, B, C} ──
fig, axes = plt.subplots(1, 4, figsize=(11, 3.0))
for ax, n in zip(axes, [3, 4, 5, 6]):
    nodes_n = list("ABCDEF"[:n])
    edges_n = list(itertools.combinations(nodes_n, 2))
    draw_graph(edges_n, nodes=nodes_n, ax=ax, title=f"K{n}: {len(edges_n)} edges = {n}·{n - 1}/2, {n - 1}-regular")
    assert regularity(nodes_n, edges_n) == n - 1 and len(edges_n) == math.comb(n, 2)
plt.tight_layout(); plt.show()

abcd_edges = [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D")]


def is_clique(subset, edges) -> bool:
    """True if every two distinct nodes of subset are joined by an edge."""
    E = as_undirected(edges)
    return all(frozenset(pair) in E for pair in itertools.combinations(subset, 2))


for size in [2, 3, 4]:
    found = ["".join(s) for s in itertools.combinations("ABCD", size) if is_clique(s, abcd_edges)]
    print(f"cliques with {size} nodes in the A–D graph:", found)
print("directed complete graph on 4 nodes: |V × V minus the loops| =",
      len([(u, v) for u in "ABCD" for v in "ABCD" if u != v]))

🔍 **What just happened?** The four drawings show $K_3$, $K_4$, $K_5$ and $K_6$ with 3, 6, 10 and 15 edges — each
$\binom n2$, and each graph $(n-1)$-regular (the `assert` would have stopped the cell otherwise). In the A–D graph every
edge is a 2-node clique, {A, B, C} is the only 3-node clique, and there is no 4-node clique (D is joined only to C).
The directed complete graph on 4 nodes has $4\cdot 3 = 12$ edges: every ordered pair of distinct nodes.

**Biology.** A clique in a protein-interaction network is a group of proteins that all bind each other — the signature
of a stable **protein complex**; module-finding algorithms search for dense, nearly complete subgraphs.

✏️ **Your turn.** How many edges does a complete graph on the roughly 20,000 human protein-coding genes have? Real
interaction maps contain a few hundred thousand interactions — what *fraction* of all possible pairs is that (the
network's **density**)?

### 📘 Lecture notes — neighbourhoods, degrees, regular and complete graphs

**Undirected.** $N_G(v) = \{u \mid \{u, v\}\in E\}$ (the neighbours of $v$);
$\operatorname{degree}_G(v) := |N_G(v)| + \mathbb 1(v\in N_G(v))$, where $\mathbb 1:\{\text{true}, \text{false}\}\to\{1, 0\}$
is an indicator function. Equivalently, the number of edges incident with $v$, a loop counting **twice**. Examples:
$E = \{\{v,a\},\{v,b\}\}$ gives $N(v) = \{a, b\}$, degree $2 + 0 = 2$; $E = \{\{v,a\},\{v,b\},\{v,v\}\}$ gives
$N(v) = \{a, b, v\}$, degree $3 + 1 = 4$. Lake City has degree 3, Los Angeles 1.

**Handshake lemma.** $\sum_v\deg(v) = 2|E|$; hence the number of odd-degree nodes is even.

**Exercises.** $a\dots g$: $\deg = 2, 4, 4, 1, 3, 4, 0$ for $a, b, c, d, e, f, g$. Multigraph: $\deg(a) = 4$,
$\deg(b) = \deg(e) = 6$, $\deg(c) = 1$, $\deg(d) = 5$ (multigraphs are not examined). Question graph A–H: $N(C) = \{B, E\}$;
degree 1: A, D, F, H; highest degree: B and E (3); $N(B)\cap N(E) = \{C\}$.

**Directed.** $N^{\text{in}}(v) = \{u\mid\langle u,v\rangle\in E\}$, $N^{\text{out}}(v) = \{u\mid\langle v,u\rangle\in E\}$,
$N(v) = N^{\text{in}}(v)\cup N^{\text{out}}(v)$ (the nodes adjacent to $v$); in-/out-degree = their sizes. A–H example: C has
in-neighbour B and out-neighbours D, E; E has in-neighbours B, C, F and out-neighbours B, F (so B, F are in both); A has
in-degree 0; G has out-degree 0.

**Regular.** $\forall v: \operatorname{degree}(v) = k$ ($d$-regular); directed: identical in- and out-degrees (a 2-regular
directed graph on 4 vertices).

**Complete graphs / cliques.** $\{u, v\}\in E$ for all distinct $u, v$; $K_n$ is $(n-1)$-regular with $\binom n2$ edges.
A clique is also a complete subgraph: {A, B, C} in the graph with edges AB, AC, BC, CD.

**Common confusions.** A loop adds 2, not 1, to the degree; a neighbourhood is a set (parallel edges count once in it);
in a directed graph $N(v)$ is a union, so a node that is both in- and out-neighbour is counted once.

> 🎓 **Exam tip.** Answer degree questions with a small table (node, neighbourhood, degree) and finish with the
> handshake check $\sum\deg = 2|E|$ (or $\sum\text{in} = \sum\text{out} = |E|$).

---
## 6 · Subgraphs and induced subgraphs

> 📍 **Where we are.** Many times we only want to look at a particular part of a graph. Take a gene regulatory
> network — a directed graph whose nodes are genes, with an edge from one gene to another if the source somehow impacts
> the expression of the sink. After an experiment we have a list of genes that are **differentially expressed** in some
> condition (their expression differs from the healthy or control case), and we want to understand the part of the
> network that involves only those genes.

A **subgraph** of $G = \langle V, E\rangle$ is a graph $G' = \langle V', E'\rangle$ with

$$V'\subseteq V \qquad\text{and}\qquad E'\subseteq\big\{\langle u, v\rangle \mid u, v\in V'\wedge\langle u, v\rangle\in E\big\}.$$

*In words:* choose a subset of the nodes, and then some of the edges that join two chosen nodes — never an edge that
leaves the chosen set. The **induced subgraph** $G[V']$ is the special case in which $E'$ *equals* that set of edges:
keep **every** edge of $G$ between chosen nodes. In the gene-expression example we almost always want the induced
subgraph — dropping an existing regulation would hide biology.

**Example.** Let $G$ have nodes {A, B, C, D} and edges {A,B}, {A,C}, {B,C}, {C,D}. Then $G'$ with nodes {B, C, D} and
edges {B,C}, {C,D} is a subgraph of $G$ (here it is even the induced subgraph on {B, C, D}). In fact, $G$ has many
different subgraphs; the next cell lists all subgraphs with exactly three nodes.

In [ ]:
# ── 6.1 All subgraphs of G with exactly 3 nodes (and the 4 induced ones), drawn as small panels ──
abcd_pos = {"A": (0, 0), "B": (1, 0.8), "C": (1, -0.6), "D": (2, 0)}


def induced_edges(subset, edges) -> list:
    """The edges of the induced subgraph G[subset]: all edges with both ends in subset."""
    return [e for e in edges if e[0] in subset and e[1] in subset]


def draw_subgraph_panel(ax, pos: dict, all_edges, chosen_nodes, chosen_edges):
    """Draw the whole graph faintly and the chosen subgraph in strong colours (tiny panels, small node letters)."""
    for (u, v) in all_edges:
        strong = (u, v) in chosen_edges
        ax.plot(*zip(pos[u], pos[v]), color="#674ea7" if strong else "0.88", lw=2 if strong else 1, zorder=1)
    for v, (x, y) in pos.items():
        ax.scatter(x, y, s=60, color="#b4a7d6" if v in chosen_nodes else "white", edgecolors="0.4", zorder=2)
        below = pos[v][1] < 0                                      # put the letter outside the drawing
        ax.text(x, y - 0.22 if below else y + 0.22, v, ha="center", va="top" if below else "bottom", fontsize=6,
                color="0.25" if v in chosen_nodes else "0.65")
    ax.set_aspect("equal"); ax.axis("off"); ax.margins(0.2)


subgraphs = []
for nodes3 in itertools.combinations("ABCD", 3):
    candidates = induced_edges(nodes3, abcd_edges)
    for r in range(len(candidates), -1, -1):                 # every subset of the candidate edges (a power set)
        for chosen in itertools.combinations(candidates, r):
            subgraphs.append((nodes3, chosen))
by_nodes = Counter(nodes3 for nodes3, _ in subgraphs)
print("3-node subgraphs per node set:", {"".join(k): v for k, v in by_nodes.items()}, "→ total", len(subgraphs))
print("induced 3-node subgraphs:", {"".join(n): ["".join(e) for e in induced_edges(n, abcd_edges)]
                                    for n in itertools.combinations("ABCD", 3)})

fig, axes = plt.subplots(4, 8, figsize=(11, 4.4))
for row, nodes3 in enumerate(itertools.combinations("ABCD", 3)):
    panels = [chosen for n3, chosen in subgraphs if n3 == nodes3]
    for col in range(8):
        if col < len(panels):
            draw_subgraph_panel(axes[row, col], abcd_pos, abcd_edges, nodes3, panels[col])
        else:
            axes[row, col].axis("off")
    axes[row, 0].set_title("nodes " + ", ".join(nodes3), fontsize=8, loc="left", pad=10)
fig.suptitle("all 18 subgraphs of G with exactly 3 nodes (one row per node set; first panel = induced subgraph)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For each of the $\binom43 = 4$ choices of three nodes, the possible edges are those of $G$
between the chosen nodes, and any subset of them may be kept — a **power set** (Module 03), so $2^{(\text{number of
candidate edges})}$ subgraphs. {A, B, C} has 3 candidate edges and $2^3 = 8$ subgraphs; {A, B, D} has only {A, B} and
2 subgraphs; {A, C, D} and {B, C, D} have 2 candidate edges each and 4 subgraphs each: $8 + 2 + 4 + 4 = 18$. In each row
the first panel (all candidate edges kept) is the **induced** subgraph; there are only 4 of those, one per node set.

### 6.2 The regulatory subnetwork of infection-responsive genes

We now do what the definition was invented for: restrict the inflammation network to the genes that truly respond to
infection in the spleen (the simulator's hidden truth; in a real study they would come from a differential-expression
test — Module 25).

In [ ]:
# ── 6.2 Induced subgraph on the genes changed in the SIMULATED spleen, and why hidden regulators matter ──
changed_spleen = infection["true_de"]["spleen"]
V_changed = sorted(set(NET_NODES) & changed_spleen)
E_changed = induced_edges(V_changed, NET_EDGES)
print(f"changed network genes: {len(V_changed)} → the induced subgraph keeps {len(E_changed)} of {len(NET_EDGES)} edges:")
print("  ", E_changed)
isolated = [g for g in V_changed if not any(g in e for e in E_changed)]
print("isolated in the induced subgraph:", isolated)

regulators = set().union(*(in_neighbours(g, NET_EDGES) for g in V_changed))
V_expanded = sorted(set(V_changed) | regulators)
E_expanded = induced_edges(V_expanded, NET_EDGES)
print("\nadding the regulators of changed genes brings in:", sorted(regulators - set(V_changed)),
      f"→ {len(V_expanded)} genes, {len(E_expanded)} edges")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
draw_graph(E_changed, nodes=V_changed, directed=True, pos=NET_POS, ax=axes[0], font_size=7,
           title="induced subgraph on the 12 changed genes")
draw_graph(E_expanded, nodes=V_expanded, directed=True, pos=NET_POS, ax=axes[1], font_size=7,
           node_colors={g: "#f4cccc" for g in regulators - set(V_changed)},
           title="… plus their regulators (red: not measured as changed)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** All 12 measured network genes respond to infection, yet the induced subgraph keeps only 4 of
the 15 edges — the interferon module STAT1 → IRF1, MX1, ISG15 and IRF1 → ISG15 — and the 8 other changed genes (the
cytokines, LYZ and the acute-phase genes) end up **isolated**. The reason is biological: their regulators NF-κB
(NFKB1) and STAT3 are not in the list of changed genes (in our simulation they are not even measured). Transcription
factors are often regulated by their *activity* rather than their mRNA level — NF-κB, for example, is held inactive in
the cytoplasm by IκB proteins until a signal triggers IκB degradation and NF-κB moves to the nucleus. Adding the
regulators of the changed genes (the union of their in-neighbourhoods) brings NFKB1 and STAT3 back and reconnects the
network: 14 genes and all 15 edges. Lesson: an induced subgraph is only as good as the node list you feed it; methods
that infer **transcription-factor activity** from the expression of their targets exist precisely for this reason.

✏️ **Your turn.** (a) Repeat the analysis for the liver (`infection["true_de"]["liver"]`). Does the induced subgraph
change? (b) Compute the subgraph induced by NFKB1 together with its out-neighbourhood. Predict the number of edges first
(hint: do not forget the feedback loops).

---
## 7 · Bipartite graphs

> 📍 **Where we are.** Some networks connect two *kinds* of objects: metabolites and the reactions that convert them,
> drugs and their protein targets, students and advisers. Edges only ever run between the kinds, never within one.

A **bipartite graph** is a graph $G = \langle A\cup B, E\rangle$ whose node set is split into two **disjoint** groups,
$A\cap B = \varnothing$, with

$$E\subseteq\big\{\{u, v\} \mid u\in A,\ v\in B\big\}.$$

*In words:* the nodes can be separated into two groups such that no edge exists within a group; every edge joins a node
of $A$ to a node of $B$. (In directed graphs, edges can go from $A$ to $B$ or from $B$ to $A$.) A node with no edges at
all is an **isolated node**: it still belongs to the graph, but plays no role in connectivity, and it may sit in either
group.

**Example.** Groups $\{A, E\}$ and $\{B, C, D, F\}$, written as a tuple:
$G = (V, E) = (\{A, E, B, C, D, F\},\ \{\{A, B\}, \{A, C\}, \{E, C\}, \{E, D\}\})$, with F isolated. (Careful: in this
example the letters A and B also name two *nodes*; below we call the groups "group 1" and "group 2".) In fact every
graph can be written as such a tuple; what makes this one bipartite is the restriction on $E$.

In [ ]:
# ── 7.1 The bipartite example: groups {A, E} and {B, C, D, F}; F is isolated ──
group1, group2 = {"A", "E"}, {"B", "C", "D", "F"}
bip_edges = [("A", "B"), ("A", "C"), ("E", "C"), ("E", "D")]
print("groups disjoint?", group1.isdisjoint(group2))
print("every edge joins the two groups?", all((u in group1) != (v in group1) for u, v in bip_edges))
print("isolated nodes:", sorted((group1 | group2) - {v for e in bip_edges for v in e}))
bip_pos = {"A": (0, 3), "E": (0, 0), "B": (1.2, 3), "C": (1.2, 2), "D": (1.2, 1), "F": (1.2, 0)}
draw_graph(bip_edges, nodes=["A", "E", "B", "C", "D", "F"], pos=bip_pos,
           node_colors={**{v: "#f4cccc" for v in group1}, **{v: "#cfe2f3" for v in group2}},
           title="group 1 (red) | group 2 (blue)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The two groups do not overlap, every one of the four edges runs from a red node to a blue node,
and F is the only isolated node. Drawn in two columns, the bipartite structure is obvious — but bipartiteness is a
property of the sets $V$ and $E$, not of the drawing, and a graph may be bipartite even when it is drawn differently.

### 7.2 Which of these graphs are bipartite?

How can we *decide* whether a graph is bipartite without guessing a split? Use a **two-colouring**: give any node colour 0,
give all its neighbours colour 1, their neighbours colour 0, and so on (a breadth-first search, Section 8). If we ever
find an edge whose two ends got the same colour, the graph is not bipartite; otherwise the two colours are the two
groups. **Theorem:** a graph is bipartite if and only if it contains no cycle of odd length. (One direction is easy:
walking around a cycle, the colours must alternate, which is impossible if the cycle has an odd number of edges — a
triangle, for instance.)

In [ ]:
# ── 7.2 A two-colouring test, applied to the five example graphs (i)–(v) ──
def two_colouring(nodes, edges):
    """Try to colour the nodes 0/1 so that every edge joins different colours (BFS from every uncoloured node).
    Returns (True, colours) on success, or (False, (u, w)) with an edge whose ends must share a colour."""
    nbrs = {v: set() for v in nodes}
    for u, w in edges:
        nbrs[u].add(w)
        nbrs[w].add(u)
    colour = {}
    for start in nodes:
        if start in colour:
            continue
        colour[start] = 0
        queue = deque([start])
        while queue:
            u = queue.popleft()
            for w in sorted(nbrs[u]):
                if w not in colour:
                    colour[w] = 1 - colour[u]
                    queue.append(w)
                elif colour[w] == colour[u]:
                    return False, (u, w)
    return True, colour


examples = {
    "(i)": ([("1", "2"), ("2", "3"), ("2", "4")], {"1": (0, 2), "2": (0, 1), "3": (-0.5, 0), "4": (0.5, 0)}),
    "(ii)": ([("1", "2"), ("2", "4"), ("4", "3"), ("3", "1")], {"1": (0, 1), "2": (1, 1), "3": (0, 0), "4": (1, 0)}),
    "(iii)": ([("1", "2"), ("2", "3"), ("3", "4"), ("1", "3")], {"1": (0, 3), "2": (-0.35, 2), "3": (0, 1), "4": (0, 0)}),
    "(iv)": ([("1", "4"), ("2", "4"), ("3", "4")], {"1": (0, 2), "2": (0, 1), "3": (0, 0), "4": (1, 0)}),
    "(v)": ([("1", "2"), ("3", "4"), ("1", "4"), ("3", "2")], {"1": (0, 1), "2": (1, 1), "3": (0, 0), "4": (1, 0)}),
}
fig, axes = plt.subplots(1, 5, figsize=(11, 3.0))
for ax, (name, (E, pos)) in zip(axes, examples.items()):
    ok, result = two_colouring(sorted(pos), E)
    if ok:
        groups = (sorted(v for v in result if result[v] == 0), sorted(v for v in result if result[v] == 1))
        print(f"{name:5s} bipartite: groups {groups[0]} | {groups[1]}")
        colours, title = {v: ("#f4cccc" if result[v] == 0 else "#cfe2f3") for v in pos}, f"{name} bipartite"
    else:
        print(f"{name:5s} NOT bipartite: the edge {result} would join two nodes of the same colour")
        colours, title = {v: "#eeeeee" for v in pos}, f"{name} not bipartite"
    draw_graph(E, pos=pos, ax=ax, node_colors=colours, title=title)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The test colours graphs (i), (ii), (iv) and (v) with two colours and prints the two groups;
for (iii) it finds an edge whose ends would need the same colour. Graph (iii) contains the triangle 1–2–3 (the long
edge 1–3 closes it): a cycle of odd length, so no split into two groups can work. All the others are bipartite, although
only (iv) and (v) are drawn in two columns. The next cell checks the claim that (i) is "the same graph" as (iv), and (ii)
the same as (v), up to renaming the nodes — the graphs are **isomorphic**.

In [ ]:
# ── 7.3 Isomorphism by brute force: is there a renaming of the nodes that maps one edge set onto the other? ──
def isomorphic(edges_1, edges_2) -> bool:
    """Try every bijection between the node sets (fine for tiny graphs: n! renamings)."""
    nodes_1 = sorted({v for e in edges_1 for v in e})
    nodes_2 = sorted({v for e in edges_2 for v in e})
    if len(nodes_1) != len(nodes_2) or len(edges_1) != len(edges_2):
        return False
    target = as_undirected(edges_2)
    for perm in itertools.permutations(nodes_2):
        rename = dict(zip(nodes_1, perm))
        if {frozenset((rename[u], rename[v])) for u, v in edges_1} == target:
            return True
    return False


E5 = {name: edges for name, (edges, _) in examples.items()}
print("(i) ≅ (iv)?", isomorphic(E5["(i)"], E5["(iv)"]), "| (ii) ≅ (v)?", isomorphic(E5["(ii)"], E5["(v)"]),
      "| (i) ≅ (ii)?", isomorphic(E5["(i)"], E5["(ii)"]), "| (iii) ≅ (ii)?", isomorphic(E5["(iii)"], E5["(ii)"]))

🔍 **What just happened?** Trying all $4! = 24$ renamings shows that (i) and (iv) are the same graph — a centre joined to
three other nodes (a "star" or "claw") — and (ii) and (v) are the same 4-cycle drawn as a square and as a "bow tie".
(i) and (ii) are not isomorphic (3 vs 4 edges), nor are (iii) and (ii) (same number of edges, but (iii) contains a
triangle). Isomorphism is why "is it bipartite?" never depends on the drawing.

### 7.4 Metabolism as a bipartite graph

A **metabolic network** has two kinds of nodes: metabolites and reactions. A metabolite is joined to every reaction that
consumes or produces it; two metabolites are never joined directly. The first steps of glycolysis give a small example
(the enzymes are hexokinase, phosphoglucose isomerase PGI, phosphofructokinase-1, aldolase and triose-phosphate
isomerase TPI). Bipartite graphs also model **assignments** — e.g. assigning each first-year student to one second-year
peer adviser: with first-years $A$ and second-years $B$, start from all possible pairs $E = A\times B$ and look for a
subgraph in which every first-year has out-degree exactly 1.

In [ ]:
# ── 7.4 The first steps of glycolysis (typed in): a bipartite metabolite–reaction graph and its projection ──
reactions = {"hexokinase": (["glucose", "ATP"], ["G6P", "ADP"]),
             "PGI": (["G6P"], ["F6P"]),
             "PFK-1": (["F6P", "ATP"], ["F1,6BP", "ADP"]),
             "aldolase": (["F1,6BP"], ["DHAP", "G3P"]),
             "TPI": (["DHAP"], ["G3P"])}
metab_edges = [(m, r) for r, (subs, prods) in reactions.items() for m in subs + prods]
metabolites = sorted({m for m, _ in metab_edges})
ok, _ = two_colouring(metabolites + list(reactions), metab_edges)
print(f"{len(metabolites)} metabolites, {len(reactions)} reactions, {len(metab_edges)} edges; bipartite? {ok}")

projection = {frozenset(p) for r, (subs, prods) in reactions.items()
              for p in itertools.combinations(subs + prods, 2)}             # metabolites sharing a reaction
proj_degree = {m: sum(m in e for e in projection) for m in metabolites}
print("metabolite graph (projection):", len(projection), "edges; degrees:",
      dict(sorted(proj_degree.items(), key=lambda kv: (-kv[1], kv[0]))))

order_m = ["glucose", "ATP", "G6P", "ADP", "F6P", "F1,6BP", "DHAP", "G3P"]
pos_m = {m: (0, -i) for i, m in enumerate(order_m)}
pos_r = {r: (2.4, -0.5 - 1.5 * i) for i, r in enumerate(reactions)}
fig, ax = plt.subplots(figsize=(5.2, 4.6))
draw_graph(metab_edges, pos={**pos_m, **pos_r}, ax=ax, font_size=7,
           node_colors={**{m: "#d9ead3" for m in metabolites}, **{r: "#fce5cd" for r in reactions}},
           title="metabolites (green) ↔ reactions (orange)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 8 metabolites and 5 reactions form a bipartite graph with 15 edges (4 + 2 + 4 + 3 + 2
metabolites per reaction; the two-colouring test confirms bipartiteness). **Projecting** onto metabolites — joining two
metabolites whenever they take part in the same reaction — gives an ordinary graph with 15 edges in which the cofactors
ATP and ADP have the highest degree (5, tied with fructose-1,6-bisphosphate), although they are only "currency"
metabolites carrying energy between reactions; G6P and F6P (degree 4) are well connected because each sits between two
reactions. In genome-scale networks currency metabolites connect almost everything, which is why metabolic network
analyses often keep the bipartite form, or remove currency metabolites, before looking for pathways.

✏️ **Your turn.** Add the reaction `"GAPDH": (["G3P", "NAD+", "Pi"], ["1,3BPG", "NADH"])` and re-run. Is the graph still
bipartite? Which metabolite's projected degree grows the most?

### 📘 Lecture notes — subgraphs and bipartite graphs

**Subgraph.** $G' = \langle V', E'\rangle$ with $V'\subseteq V$ and $E'\subseteq\{\langle u,v\rangle\mid u,v\in V'\wedge\langle u,v\rangle\in E\}$:
choose a subset of nodes and edges associated only with the chosen nodes. **Induced subgraph:** $E'$ *equal* to that set.
Motivation: in bioinformatics we often do not examine the entire graph but the subgraph of regulatory relationships
among genes that are differentially expressed under specific conditions (disease, treatment). Example: $G$ with nodes
{A,B,C,D} and edges {A,B},{A,C},{B,C},{C,D}; $G' = \langle\{B,C,D\}, \{\{B,C\},\{C,D\}\}\rangle$ is a subgraph; $G$ has 18
subgraphs with exactly 3 nodes (8 on ABC, 2 on ABD, 4 on ACD, 4 on BCD) and 4 induced ones.

**Bipartite graph.** $G = \langle A\cup B, E\rangle$ with $A\cap B = \varnothing$ and $E\subseteq\{\{u,v\}\mid u\in A, v\in B\}$
(directed: edges may go either way between the groups): no edge within a group. Isolated nodes (no edges) still belong
to the graph. Example: $A = \{A, E\}$, $B = \{B, C, D, F\}$, $E = \{\{A,B\},\{A,C\},\{E,C\},\{E,D\}\}$, F isolated. Of the
five example graphs only (iii) is not bipartite ((i) ≅ (iv), (ii) ≅ (v)); a graph is bipartite iff it has no odd cycle.
Uses: assignments (students ↔ advisers), metabolites ↔ reactions, drugs ↔ targets.

**Common confusion.** Bipartiteness does not depend on the drawing; and an induced subgraph keeps *all* edges among the
chosen nodes, while a general subgraph may drop some.

> 🎓 **Exam tip.** To show a graph is bipartite, exhibit the two groups; to show it is not, exhibit an odd cycle.

---
## 8 · Connectivity and breadth-first search

> 📍 **Where we are.** Can every gene's signal reach every other gene? Are there separate modules? These are questions
> about **connectivity**, and the formal definition is a nested-quantifier statement of the kind Module 02 taught you
> to read.

A graph $G$ is **connected** if

$$\forall u, v\in V:\ \exists\,\langle u, z_1, z_2, \dots, z_k, v\rangle\in V^*:\ \langle u, z_1\rangle, \langle z_1, z_2\rangle, \dots, \langle z_k, v\rangle\in E .$$

*In words:* for **every** pair of nodes there **exists** a finite sequence of nodes — an element of the Kleene star $V^*$
of Module 04 — that starts at $u$, ends at $v$ and whose consecutive pairs are all edges: the graph is connected if for
every pair of nodes there is a path between them. (For undirected graphs read the edges as $\{\cdot, \cdot\}$.)

A **connected component** is a connected subgraph that is **maximal** — it cannot be enlarged by adding nodes and still
be connected. A graph that is already connected has only one connected component.

⚠️ **Common confusion.** "A connected component is a subgraph of $G$ that is connected" is not enough: a single node,
or any single edge, is a connected subgraph. The word *maximal* is essential — components are the separate "islands" of
the graph, and every node belongs to exactly one of them (the components partition $V$, Module 03).

**Breadth-first search (BFS).** To find everything reachable from a start node $s$: visit $s$ (distance 0), then all its
neighbours (distance 1), then all *their* unvisited neighbours (distance 2), and so on, using a first-in-first-out queue.
Every node and every edge is handled at most a constant number of times, so BFS runs in $O(|V| + |E|)$ time
(Module 05). BFS also gives the **shortest path length** (number of edges) from $s$ to every reachable node, and
repeating it from an unvisited node finds all connected components.

In [ ]:
# ── 8.1 BFS, distances and connected components: the connected and the disconnected example ──
def adjacency_lists(nodes, edges, directed: bool = False) -> dict:
    """node → set of nodes one edge away (in both directions for an undirected graph)."""
    nbrs = {v: set() for v in nodes}
    for u, w in edges:
        nbrs[u].add(w)
        if not directed:
            nbrs[w].add(u)
    return nbrs


def bfs(start, nbrs: dict) -> tuple:
    """Breadth-first search from start: distance (number of edges) and BFS parent of every reachable node."""
    dist, parent, queue = {start: 0}, {start: None}, deque([start])
    while queue:
        u = queue.popleft()
        for w in sorted(nbrs[u]):
            if w not in dist:
                dist[w] = dist[u] + 1
                parent[w] = u
                queue.append(w)
    return dist, parent


def connected_components(nodes, edges) -> list:
    """The connected components (as sorted node lists) of an undirected graph, by repeated BFS."""
    nbrs, seen, components = adjacency_lists(nodes, edges), set(), []
    for v in nodes:
        if v not in seen:
            reached, _ = bfs(v, nbrs)
            components.append(sorted(reached))
            seen |= set(reached)
    return components


nodes6 = list("abcdef")
left_edges = [("a", "b"), ("a", "c"), ("b", "d"), ("c", "d"), ("b", "e"), ("d", "f"), ("e", "f")]
right_edges = [("a", "b"), ("a", "c"), ("c", "d"), ("e", "f")]
conn_pos = {"a": (0, 0), "b": (1, 1), "c": (1, -1), "d": (2, 0), "e": (3.4, 1), "f": (3.4, 0)}
palette = ["#cfe2f3", "#f4cccc", "#d9ead3", "#fff2cc"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (name, E) in zip(axes, [("left", left_edges), ("right", right_edges)]):
    comps = connected_components(nodes6, E)
    dist_a, _ = bfs("a", adjacency_lists(nodes6, E))
    every_pair = all(v in bfs(u, adjacency_lists(nodes6, E))[0] for u in nodes6 for v in nodes6)
    print(f"{name}: components {comps} → connected? {len(comps) == 1} (∀u,v ∃ path: {every_pair}); "
          f"distances from a: {dist_a}")
    colours = {v: palette[i] for i, comp in enumerate(comps) for v in comp}
    draw_graph(E, nodes=nodes6, pos=conn_pos, ax=ax, node_colors=colours,
               title=f"{name}: {len(comps)} connected component{'s' if len(comps) > 1 else ''}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The left graph is connected: BFS from $a$ reaches all six nodes, at distances
$b, c = 1$, $d, e = 2$, $f = 3$, and the brute-force check of "∀ u, v ∃ path" over all 36 ordered pairs is true. The
right graph is not: BFS from $a$ reaches only $a, b, c, d$, and a second search from $e$ finds the component $\{e, f\}$ —
two islands, coloured differently. The quantifier check fails as soon as one pair (say $a$ and $e$) has no path.

### 8.2 Reachability in the inflammation network

In a directed graph, "connected" splits in two. Ignoring directions gives the **weakly connected components**; following
arrows answers *which genes can a regulator influence?* (BFS along out-edges) and *which genes can influence a target?*
(BFS along in-edges).

In [ ]:
# ── 8.2 Weak components and directed reachability in the inflammation network ──
weak = connected_components(NET_NODES, NET_EDGES)
print("weakly connected components (directions ignored):")
for comp in weak:
    print("  ", comp)

downstream, _ = bfs("NFKB1", adjacency_lists(NET_NODES, NET_EDGES, directed=True))
print("\ngenes NF-κB can influence (directed distance):", {g: d for g, d in sorted(downstream.items(), key=lambda kv: kv[1]) if g != "NFKB1"})
reversed_edges = [(w, u) for (u, w) in NET_EDGES]
upstream, _ = bfs("HAMP", adjacency_lists(NET_NODES, reversed_edges, directed=True))
print("genes that can influence HAMP (distance against the arrows):",
      {g: d for g, d in sorted(upstream.items(), key=lambda kv: kv[1]) if g != "HAMP"})
print("can STAT1 influence SAA1?", "SAA1" in bfs("STAT1", adjacency_lists(NET_NODES, NET_EDGES, directed=True))[0])

🔍 **What just happened?** Ignoring directions, the network splits into two weakly connected components: the
NF-κB/IL-6/STAT3 component (10 genes, cytokines and acute-phase genes) and the STAT1 interferon component (STAT1, IRF1,
MX1, ISG15). Following arrows from NF-κB reaches its five direct targets at distance 1, STAT3 at distance 2 (via IL6)
and the acute-phase genes at distance 3 — and NFKB1 itself is reachable again through the feedback loops (so BFS lists
only the other genes). Going *against* the arrows from the iron hormone hepcidin (HAMP) finds its regulators STAT3 (1),
IL6 (2), NFKB1 (3) and TNF, IL1B (4): a chain from bacterial sensing to iron withholding. STAT1 cannot influence SAA1 —
they are in different components of this cartoon network.

### 🌉 Bridge — co-expression modules are connected components

Large biological networks are often built *from data*: connect two genes when their expression profiles across samples
are strongly correlated (a **co-expression network**); the connected components and dense regions are interpreted as
modules of co-regulated genes — immune response, ribosome, cell cycle, extracellular matrix. We build one from the
SIMULATED infection experiment, using the 12 infection-pathway genes and the 12 housekeeping genes and the correlation
coefficient $r$ across the 24 samples (Modules 07 and 21 define $r$ properly; here it is just a similarity score
between −1 and +1). An edge joins two genes if $|r|\ge$ a threshold.

In [ ]:
# ── 8.3 A co-expression network from the SIMULATED infection data; modules = connected components ──
named_genes = infection["genes"][:24]                    # the 12 infection-pathway genes, then the 12 housekeeping genes
columns = [infection["genes"].index(g) for g in named_genes]
R = np.corrcoef(infection["X"][:, columns].T)            # 24 × 24 gene–gene correlations across the 24 samples
for threshold in [0.7, 0.8]:
    co_edges = [(named_genes[i], named_genes[j]) for i in range(24) for j in range(i + 1, 24) if abs(R[i, j]) >= threshold]
    comps = connected_components(named_genes, co_edges)
    modules = [c for c in comps if len(c) > 1]
    print(f"|r| ≥ {threshold}: {len(co_edges)} edges, modules: {modules}, isolated genes: {sum(len(c) == 1 for c in comps)}")
    if threshold == 0.7:
        edges_07, comps_07 = co_edges, comps

colours = {g: (palette[1] if len(c) > 1 else palette[0]) for c in comps_07 for g in c}
module_genes, other_genes = named_genes[:12], named_genes[12:]
co_pos = {g: (1.6 * x, 1.6 * y) for g, (x, y) in circle_layout(module_genes).items()}          # module on a circle
co_pos.update({g: (2.9 + 0.75 * (i % 3), 1.2 - 0.8 * (i // 3)) for i, g in enumerate(other_genes)})   # others in a grid
fig, ax = plt.subplots(figsize=(7.2, 4.4))
draw_graph(edges_07, nodes=named_genes, pos=co_pos, ax=ax, node_colors=colours, font_size=6,
           title="SIMULATED co-expression network, |r| ≥ 0.7:\none infection module (red), housekeeping genes isolated (blue)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the threshold $|r|\ge 0.7$ the network has 39 edges and exactly one non-trivial
component — all 12 infection-pathway genes, which rise together in infected samples — while the 12 housekeeping genes
(stable by design) are isolated. Raising the threshold to 0.8 leaves 27 edges and splits the module: IL6 and ISG15 form a
separate pair while the other nine stay together, and IRF1 becomes isolated. The modules a co-expression analysis
reports therefore depend on an arbitrary threshold — one reason why methods for network modules use soft thresholds or
clustering (BiologyToML [Module 12](../../../../ml/tutorial/BiologyToML/notebooks/12_clustering_kmeans_hierarchical.ipynb), and Gaussian graphical models in
[Module 18](../../../../ml/tutorial/BiologyToML/notebooks/18_gaussian_graphical_models.ipynb), which separate direct from indirect correlations).

✏️ **Your turn.** Use `bfs` on the undirected version of the co-expression graph at $|r|\ge 0.7$ to find the largest
shortest-path distance between two genes of the module (the module's **diameter**). Then try the threshold 0.6.

---
## 9 · Trees

> 📍 **Where we are.** The most important special graphs in biology are **trees**: phylogenies, taxonomies, lineage trees
> of dividing cells, hierarchical clusterings, decision trees. A tree is the most economical way to connect nodes.

A **tree** is a graph that is connected and has exactly $|V| - 1$ edges. One consequence is that a tree has **no
cycles**. A **cycle** is a special type of path that ends at the node where it started,

$$\langle v_1, v_2, \dots, v_{k-1}, v_1\rangle .$$

*In words:* a cycle is a closed walk along edges that returns to its start without reusing an edge. ⚠️ In an undirected
graph a cycle needs at least **three distinct** nodes ($k - 1\ge 3$): going from $v_1$ to $v_2$ and straight back along
the same edge, $\langle v_1, v_2, v_1\rangle$, is not a cycle (otherwise every graph with an edge — including every tree —
would contain one). In a directed graph, $u\to v\to u$ (two different edges) is a genuine cycle of length 2 — like the
NFKB1 ⇄ TNF feedback loop — and a self-loop is a cycle of length 1.

**Equivalent characterisations** (for a graph with $n$ nodes): (1) connected with $n - 1$ edges; (2) connected and without
cycles; (3) without cycles and with $n - 1$ edges; (4) between any two nodes there is exactly **one** path. Removing any
edge of a tree disconnects it; adding any edge creates a cycle.

**Leaves and internal nodes.** In a tree, the nodes with only one neighbour,

$$\operatorname{Leaves}(G) := \{v\in V \mid \operatorname{degree}(v) = 1\},$$

are called **leaves**; the nodes that are not leaves, $V\setminus\operatorname{Leaves}(G)$, are **internal nodes**.
*In words:* leaves are the tips of the tree; everything else is a branching or passing point.

In [ ]:
# ── 9.1 Recognising trees: connected + |V| − 1 edges ⇒ no cycle; leaves and internal nodes ──
def has_cycle(nodes, edges) -> bool:
    """Undirected graph: True if some edge joins two nodes that are already connected (union–find)."""
    parent = {v: v for v in nodes}

    def root(v):
        while parent[v] != v:
            parent[v] = parent[parent[v]]
            v = parent[v]
        return v

    for u, w in edges:
        ru, rw = root(u), root(w)
        if ru == rw:
            return True
        parent[ru] = rw
    return False


def is_tree(nodes, edges) -> bool:
    """A tree is connected and has exactly |V| − 1 edges."""
    return len(connected_components(nodes, edges)) == 1 and len(edges) == len(nodes) - 1


def leaves_of(nodes, edges) -> list:
    """Leaves(G) = {v | degree(v) = 1}."""
    return [v for v in nodes if degree(v, edges) == 1]


samples = {
    "claw (4 nodes)": [("r", "m"), ("m", "x"), ("m", "y")],
    "11 nodes": [("r", "a"), ("r", "b"), ("a", "c"), ("a", "d"), ("b", "e"), ("b", "f"), ("c", "g"), ("c", "h"),
                 ("e", "i"), ("e", "j")],
    "path (4 nodes)": [("p1", "p2"), ("p2", "p3"), ("p3", "p4")],
    "leaves example": [("r", "l1"), ("r", "n2"), ("n2", "l2"), ("n2", "n3"), ("n3", "l3"), ("n3", "n4"), ("n3", "l4"),
                       ("n4", "l5")],
    "NOT a tree: triangle + isolated node": [("x", "y"), ("y", "z"), ("x", "z")],
}
extra_nodes = {"NOT a tree: triangle + isolated node": ["w"]}
rows = []
for name, E in samples.items():
    V = sorted({v for e in E for v in e} | set(extra_nodes.get(name, [])))
    rows.append({"graph": name, "|V|": len(V), "|E|": len(E), "connected": len(connected_components(V, E)) == 1,
                 "|E| = |V|−1": len(E) == len(V) - 1, "has cycle": has_cycle(V, E), "tree": is_tree(V, E),
                 "leaves": len(leaves_of(V, E))})
print(pd.DataFrame(rows).to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(11, 3.2))
for ax, name in zip(axes, ["claw (4 nodes)", "11 nodes", "path (4 nodes)", "leaves example"]):
    E = samples[name]
    V = sorted({v for e in E for v in e})
    root_node = "p1" if name.startswith("path") else "r"
    draw_graph(E, pos=tree_layout(E, root_node), ax=ax, font_size=7,
               node_colors={v: ("#6aa84f" if v in leaves_of(V, E) else "#d9ead3") for v in V},
               title=f"{name}: {len(V)} nodes, {len(E)} edges")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The four sample trees have 4, 11, 4 and 9 nodes and always exactly one edge fewer; all are
connected and none has a cycle. The leaves (dark green) are the degree-1 nodes: 3, 6, 2 and 5 of them. The last row of
the table is the warning example: a triangle plus an isolated node has $3 = |V| - 1$ edges, yet it is disconnected and
has a cycle — so "$|V| - 1$ edges" alone does not make a tree; connectivity is needed too (or, equivalently, the absence
of cycles).

### 9.2 Rooted trees

Many times — though not always — we deal with **rooted trees**: one node $r\in V$ is designated as the **root**. The
neighbours of the root are its **children**; the children's neighbours (excluding their parent) are their children, and
so on. This defines a **parent–child hierarchy** throughout the tree, and **levels** (the distance from the root). All
nodes below a node $v$, with the graph they induce, form the **subtree** rooted at $v$ — slightly different from a
general subgraph, since it is defined only for rooted trees and is always an induced subgraph hanging from $v$.

⚠️ In a rooted tree, "leaf" means "no children". This agrees with "degree 1" except in one corner case: a root with a
single child also has degree 1 but is not called a leaf.

In [ ]:
# ── 9.2 Root, parents, children, levels, leaves and subtrees (the two rooted-tree examples) ──
def rooted_tree_info(edges, root) -> dict:
    """Parent, children and level of every node of a tree rooted at `root` (BFS from the root)."""
    nodes = sorted({v for e in edges for v in e})
    level, parent = bfs(root, adjacency_lists(nodes, edges))
    children = {v: sorted(w for w in nodes if parent.get(w) == v) for v in nodes}
    return {"parent": parent, "children": children, "level": level}


def subtree_nodes(v, children: dict) -> list:
    """v together with all of its descendants."""
    found = [v]
    for c in children[v]:
        found += subtree_nodes(c, children)
    return found


tree9 = [("A", "B"), ("A", "C"), ("C", "D"), ("C", "E"), ("E", "F"), ("E", "G"), ("E", "H"), ("G", "I")]
info9 = rooted_tree_info(tree9, "A")
print("9-node tree: root A | E's parent:", info9["parent"]["E"], "| E's children:", info9["children"]["E"],
      "| level of I:", info9["level"]["I"])
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
pos9 = tree_layout(tree9, "A")
highlights = [({"A"}, "root"), ({"C"}, "E's parent"), (set(info9["children"]["E"]), "E's children")]
for ax, (chosen, title) in zip(axes, highlights):
    colours = {v: ("#6aa84f" if v in chosen else "#d9ead3") for v in pos9}
    if title != "root":
        colours["E"] = "#9fc5e8"
    draw_graph(tree9, pos=pos9, ax=ax, node_colors=colours, title=title)
plt.tight_layout(); plt.show()

tree13 = [("A", "B"), ("A", "C"), ("B", "D"), ("B", "E"), ("B", "F"), ("E", "H"), ("E", "I"), ("I", "K"),
          ("I", "L"), ("I", "M"), ("C", "G"), ("G", "J")]
info13 = rooted_tree_info(tree13, "A")
nodes13 = sorted(info13["level"])
leaves13 = [v for v in nodes13 if not info13["children"][v]]
print("13-node tree: root A")
print("  leaves (no children):", leaves13, "| internal nodes:", [v for v in nodes13 if v not in leaves13])
print("  parent of B:", info13["parent"]["B"], "| children of B:", info13["children"]["B"],
      "| parent of A:", info13["parent"]["A"], "| children of A:", info13["children"]["A"])
print("  levels:", {lvl: [v for v in nodes13 if info13["level"][v] == lvl] for lvl in range(max(info13["level"].values()) + 1)})
print("  subtree rooted at E:", subtree_nodes("E", info13["children"]),
      "| leaves by the degree-1 rule:", sorted(leaves_of(nodes13, tree13)))

🔍 **What just happened?** In the 9-node tree rooted at A, E's parent is C and E's children are F, G and H (the three
panels highlight the root, E's parent and E's children, with E in blue); I is at level 4. For the 13-node tree the code
reproduces the textbook answers: root A; leaves {D, F, H, J, K, L, M}; internal nodes {A, B, C, E, G, I}; parent of B is
A and children of B are {D, E, F}; A has no parent (`None`) and children {B, C}. The levels list the tree generation by
generation (level 4 holds K, L, M), the subtree rooted at E contains E, H, I, K, L, M, and here the degree-1 rule finds
the same leaves because the root A has two children.

### 9.3 $k$-ary trees

Up to now there was no limit on the number of neighbours (children) a node could have, and in general this is true, but
many times we want to know some general property of a tree. A **$k$-ary tree** is one where every internal node has
degree at most $k + 1$:

$$\forall v\in V,\ v \text{ internal}:\ |N(v)|\le k + 1 .$$

*In words:* each internal node has at most $k$ "downward" neighbours plus one "upward" neighbour. It makes more sense in
rooted trees, where it means that each internal node has **at most $k$ children**; a rooted **binary** tree ($k = 2$) has
at most 2 children per node. A rooted tree is a **full $k$-ary tree** if every internal node has **exactly** $k$ children.
In a full $k$-ary tree with $i$ internal nodes, every node except the root is the child of exactly one internal node, so

$$n = k\,i + 1 \ \text{ nodes}, \qquad \ell = n - i = (k-1)\,i + 1 \ \text{ leaves}.$$

*In words:* counting the children of all internal nodes counts every node except the root once.

In [ ]:
# ── 9.3 The four textbook trees T1–T4: which are full k-ary trees? ──
def draw_tree_dots(ax, edges, root, title: str = ""):
    """Draw a rooted tree in textbook style: small black dots, no labels (positions from tree_layout)."""
    pos = tree_layout(edges, root)
    for u, v in edges:
        ax.plot(*zip(pos[u], pos[v]), color="0.2", lw=1, zorder=1)
    xs, ys = zip(*pos.values())
    ax.scatter(xs, ys, s=16, color="k", zorder=2)
    ax.margins(0.08); ax.axis("off"); ax.set_title(title)


T = {"T1": [(0, 1), (0, 2), (1, 3), (1, 4), (4, 5), (4, 6)],
     "T2": [(0, 1), (0, 2), (0, 3), (1, 4), (1, 5), (1, 6), (5, 7), (5, 8), (5, 9), (2, 10), (2, 11), (2, 12)],
     "T3": [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5)] + [(2, c) for c in range(6, 11)] + [(5, c) for c in range(11, 16)],
     "T4": [(0, 1), (0, 2), (1, 3), (1, 4), (1, 5), (4, 6), (4, 7), (2, 8), (2, 9), (2, 10)]}
fig, axes = plt.subplots(1, 4, figsize=(11, 3.0), gridspec_kw={"width_ratios": [1, 1.4, 1.8, 1.2]})
for ax, (name, E) in zip(axes, T.items()):
    info = rooted_tree_info(E, 0)
    internal = [v for v, ch in info["children"].items() if ch]
    child_counts = sorted({len(info["children"][v]) for v in internal})
    n, i = len(info["level"]), len(internal)
    full_k = child_counts[0] if len(child_counts) == 1 else None
    check = f"n = {full_k}·{i} + 1 = {full_k * i + 1}" if full_k else "not full k-ary"
    print(f"{name}: {n:2d} nodes, {i} internal, children per internal node {child_counts} → "
          f"{'full ' + str(full_k) + '-ary' if full_k else 'not a full k-ary tree for any k'}; {check}; "
          f"leaves {n - i}; max |N(v)| of an internal node = {max(len(neighbourhood(v, E)) for v in internal)}")
    draw_tree_dots(ax, E, 0, title=f"{name}: " + (f"full {full_k}-ary" if full_k else "not full k-ary"))
plt.tight_layout(); plt.show()

🔍 **What just happened?** T1 is a **full binary tree** — each of its 3 internal vertices has two children — with
$7 = 2\cdot 3 + 1$ nodes and 4 leaves. T2 is a **full 3-ary tree** (4 internal vertices, each with three children):
$13 = 3\cdot 4 + 1$ nodes, 9 leaves. In T3 each of the 3 internal vertices has five children, so T3 is a **full 5-ary
tree** with $16 = 5\cdot 3 + 1$ nodes. T4 is **not a full $k$-ary tree for any $k$**, because some internal vertices have
two children and others three. The last column checks the unrooted definition: an internal node of a $k$-ary tree has
at most $k + 1$ neighbours (T1: 3, T2: 4, T3: 6 for the internal non-root nodes; the root has one neighbour fewer).

### 9.4 Undirected trees, directed trees and polytrees

Tree-structured graphs also appear as directed graphs (and as Bayesian networks, Module 20): (a) an **undirected tree**;
(b) a **directed tree**, in which every node except the root has exactly one parent; (c) a **directed polytree**, a
directed graph in which some nodes may have several parents but there is still exactly one path between any pair of
nodes when directions are ignored (there can be several nodes without parents). When a polytree is converted to its
**moralised** undirected form — "marry" the parents of every node by joining them with an edge, then drop the
directions — the resulting graph contains loops, which matters for probabilistic inference.

In [ ]:
# ── 9.4 Three tree-structured graphs; moralising a polytree creates a cycle ──
undirected_tree = [("c", "a1"), ("c", "a2"), ("c", "a3"), ("c", "a4")]
directed_tree = [("r", "x"), ("r", "y"), ("x", "x1"), ("x", "x2"), ("y", "y1"), ("y", "y2")]
polytree = [("p1", "c"), ("p2", "c"), ("c", "k1"), ("c", "k2")]


def moralise(directed_edges) -> list:
    """Join every pair of parents of each node, then forget directions (returns undirected pairs)."""
    nodes = {v for e in directed_edges for v in e}
    married = [tuple(sorted(pair)) for v in nodes for pair in itertools.combinations(sorted(parents(v, directed_edges)), 2)]
    return sorted(set(tuple(sorted(e)) for e in directed_edges) | set(married))


for name, E in [("directed tree", directed_tree), ("polytree", polytree)]:
    V = sorted({v for e in E for v in e})
    max_parents = max(len(parents(v, E)) for v in V)
    moral = moralise(E)
    print(f"{name}: underlying undirected graph is a tree? {is_tree(V, E)} | max number of parents {max_parents} | "
          f"moral graph: {len(moral)} edges on {len(V)} nodes, has a cycle? {has_cycle(V, moral)}")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.0))
draw_graph(undirected_tree, ax=axes[0], pos={"c": (0, 0), "a1": (-1, 1), "a2": (1, 1), "a3": (-1, -1), "a4": (1, -1)},
           title="(a) an undirected tree")
draw_graph(directed_tree, directed=True, ax=axes[1], pos=tree_layout(directed_tree, "r"), title="(b) a directed tree")
draw_graph(polytree, directed=True, ax=axes[2], pos={"p1": (-1, 1), "p2": (1, 1), "c": (0, 0), "k1": (-1, -1), "k2": (1, -1)},
           title="(c) a directed polytree")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both directed examples are trees once directions are ignored, but in the directed tree every
node has at most one parent, while in the polytree the centre node $c$ has two parents $p_1$ and $p_2$. Moralising the
directed tree changes nothing (no node has two parents to marry): 6 edges on 7 nodes, still a tree. Moralising the
polytree adds the edge $p_1$–$p_2$: 5 edges on 5 nodes and a cycle $p_1$–$c$–$p_2$–$p_1$. This is why exact inference
algorithms for Bayesian networks treat polytrees specially (BiologyToML
[Module 17](../../../../ml/tutorial/BiologyToML/notebooks/17_bayesian_networks_inference_learning.ipynb)).

✏️ **Your turn.** (a) Draw (on paper) all trees with 4 unlabelled nodes — there are two shapes. Which one is the claw and
which the path? (b) A full binary tree has 50 leaves. How many internal nodes and how many nodes does it have?

### 📘 Lecture notes — connectivity and trees

**Connected.** $\forall u, v\in V: \exists\langle u, z_1, \dots, z_k, v\rangle\in V^*: \langle u, z_1\rangle, \dots, \langle z_k, v\rangle\in E$
— for every pair of nodes there is a path between them. A **connected component** is a (maximal) connected subgraph; a
connected graph has one component. BFS finds components and shortest path lengths in $O(|V| + |E|)$.

**Tree.** A graph that is (fully) connected and has exactly $|V| - 1$ edges; consequently it has no cycles. A **cycle**
is a path $\langle v_1, v_2, \dots, v_{k-1}, v_1\rangle$ that ends where it started (in undirected graphs with at least
three distinct nodes and no repeated edge). Sample trees: 4 nodes/3 edges, 11 nodes/10 edges, 4 nodes/3 edges.

**Leaves.** $\operatorname{Leaves}(G) := \{v\in V \mid \operatorname{degree}(v) = 1\}$; internal nodes $V\setminus\operatorname{Leaves}(G)$.

**Rooted trees.** A designated root $r$; its neighbours are its children, their other neighbours are their children,
… — a parent–child hierarchy with levels; subtrees are the induced subgraphs below a node. Example (13 nodes): root A,
leaves {D, F, H, J, K, L, M}, internal {A, B, C, E, G, I}, parent of B: A, children of B: {D, E, F}, parent of A: none,
children of A: {B, C}.

**$k$-ary trees.** $\forall v$ internal: $|N(v)|\le k + 1$; in rooted trees at most $k$ children (binary: $k = 2$). Full
$k$-ary: exactly $k$ children per internal node ($n = ki + 1$). T1 full binary, T2 full 3-ary, T3 full 5-ary, T4 not full
$k$-ary for any $k$ (internal nodes with 2 and with 3 children).

**Tree-structured graphs.** Undirected tree; directed tree (one parent per non-root node); polytree (several parents
allowed, still exactly one path between any pair when directions are ignored; its moral graph contains loops).

> 🎓 **Exam tip.** To prove a graph is a tree, check *two* things (connected and $|E| = |V| - 1$, or connected and acyclic);
> for rooted trees, list parent/children per node and remember that leaves are the nodes without children.

---
## 10 · Phylogenies and spanning trees

> 📍 **Where we are.** Trees are how biology draws history. In this section we read a phylogeny, build one from a
> distance matrix, and see how to extract a tree from any connected graph.

**Phylogeny.** In biology we often use trees to show the relatedness of species, individuals or strains, and we call them
**phylogenies**. We usually label the **leaves** with known examples (extant species, sequenced strains, …), while the
**internal nodes** represent (probably unknown) **ancestors**. Many times we also add **edge weights** that encode
evolutionary distance (for example, expected substitutions per site). In a rooted phylogeny, species that are more
closely related share a common ancestor farther from the root (lower in the drawing). So in the phylogeny of human,
chimpanzee and mouse, human and chimpanzee share an ancestor C, and only meet the mouse at the root A — the common
ancestor of all three.

In [ ]:
# ── 10.1 The human–chimpanzee–mouse phylogeny with illustrative branch lengths; distances along the unique path ──
phylo_edges = [("A", "B"), ("A", "C"), ("C", "D"), ("C", "E")]
phylo_names = {"A": "root (common ancestor of all)", "B": "mouse", "C": "ancestor of human & chimp", "D": "human",
               "E": "chimpanzee"}
branch = {("A", "B"): 0.40, ("A", "C"): 0.36, ("C", "D"): 0.006, ("C", "E"): 0.006}    # illustrative, subst./site
info_p = rooted_tree_info(phylo_edges, "A")
leaves_p = [v for v in info_p["level"] if not info_p["children"][v]]
print("leaves (observed species):", [phylo_names[v] for v in leaves_p],
      "| internal (ancestors):", [phylo_names[v] for v in info_p["level"] if v not in leaves_p])


def tree_path(u, v, info) -> list:
    """The unique path between two nodes of a rooted tree: climb from both to their lowest common ancestor."""
    up_u, x = [u], u
    while info["parent"][x] is not None:
        x = info["parent"][x]
        up_u.append(x)
    up_v, y = [v], v
    while y not in up_u:
        y = info["parent"][y]
        up_v.append(y)
    return up_u[:up_u.index(y) + 1] + up_v[-2::-1]


for u, v in [("D", "E"), ("D", "B"), ("E", "B")]:
    p = tree_path(u, v, info_p)
    print(f"{phylo_names[u]} – {phylo_names[v]}: path {'–'.join(p)}, distance = {path_weight(tuple(p), branch):.3f}")
draw_graph(phylo_edges, pos=tree_layout(phylo_edges, "A"), edge_labels=branch,
           node_colors={"B": "#d9d2e9", "D": "#fff2cc", "E": "#f4cccc", "C": "#d9ead3", "A": "#cfe2f3"},
           title="A = root, B = mouse, C = human–chimp ancestor, D = human, E = chimpanzee")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The leaves are the three observed species and the internal nodes the two hypothetical
ancestors. Because a tree has exactly one path between two nodes, the evolutionary distance between two species is the
sum of the branch lengths along that path: human–chimpanzee $0.006 + 0.006 = 0.012$, while human–mouse and
chimpanzee–mouse both go through the root, $0.006 + 0.36 + 0.40 = 0.766$ (illustrative values in expected substitutions per
site, roughly the scale of neutral DNA divergence). The lowest common ancestor of human and chimpanzee is C; that of
human and mouse is the root.

### 10.2 Building a phylogeny from distances: UPGMA

Given a matrix of pairwise distances between species, **UPGMA** (unweighted pair-group method with arithmetic mean;
Sokal & Michener 1958) builds a rooted tree bottom-up: merge the two closest clusters; the distance from the new cluster
to any other cluster is the **average** of the distances between their members; repeat until one cluster is left. A
merge at distance $d$ is drawn at height $d/2$ (each lineage contributes half of the distance). UPGMA is exactly
**average-linkage hierarchical clustering**. The distances below are typed in and illustrative (percent sequence
divergence of the right order of magnitude, not measured values).

In [ ]:
# ── 10.2 UPGMA on an illustrative distance matrix of five species; check against SciPy's average linkage ──
from scipy.spatial.distance import squareform

SPECIES = ["human", "chimpanzee", "gorilla", "mouse", "rat"]
D_SPECIES = np.array([[0.0, 1.2, 1.6, 30.0, 30.6],
                      [1.2, 0.0, 1.7, 30.2, 30.8],
                      [1.6, 1.7, 0.0, 30.4, 31.0],
                      [30.0, 30.2, 30.4, 0.0, 8.0],
                      [30.6, 30.8, 31.0, 8.0, 0.0]])        # illustrative divergences (%), symmetric


def upgma(names: list, D: np.ndarray) -> tuple:
    """UPGMA: repeatedly merge the two closest clusters; new distances are size-weighted averages.
    Returns the tree as nested tuples and the list of merges (cluster 1, cluster 2, distance, height = d/2)."""
    clusters = {name: (name, 1) for name in names}                     # label → (subtree, number of species)
    dist = {frozenset((a, b)): D[i, j] for i, a in enumerate(names) for j, b in enumerate(names) if i < j}
    merges = []
    while len(clusters) > 1:
        pair = min(dist, key=dist.get)
        a, b = sorted(pair)
        d_ab = dist.pop(pair)
        (tree_a, n_a), (tree_b, n_b) = clusters.pop(a), clusters.pop(b)
        new = f"({a}, {b})"
        for c in clusters:                                              # average distance to every other cluster
            dist[frozenset((new, c))] = (n_a * dist.pop(frozenset((a, c))) + n_b * dist.pop(frozenset((b, c)))) / (n_a + n_b)
        clusters[new] = ((tree_a, tree_b), n_a + n_b)
        merges.append((a, b, d_ab, d_ab / 2))
    return next(iter(clusters.values()))[0], merges


tree_upgma, merges = upgma(SPECIES, D_SPECIES)
for a, b, d, h in merges:
    print(f"merge {a:28s} + {b:22s} at distance {d:6.3f} → height {h:6.3f}")
print("UPGMA tree:", tree_upgma)
Z = hierarchy.linkage(squareform(D_SPECIES), method="average")
print("SciPy average-linkage merge distances:", np.round(Z[:, 2], 3), "| same as ours?",
      np.allclose(sorted(Z[:, 2]), sorted(m[2] for m in merges)))

Z_heights = Z.copy()
Z_heights[:, 2] = Z[:, 2] / 2                                   # draw merges at height d/2
fig, ax = plt.subplots(figsize=(6.4, 3.4))
hierarchy.dendrogram(Z_heights, labels=SPECIES, ax=ax, color_threshold=0, above_threshold_color="k")
ax.set_ylabel("height = distance / 2 (illustrative %)"); ax.set_title("UPGMA tree of five species (illustrative distances)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** UPGMA first joins human and chimpanzee (distance 1.2, height 0.6), then adds the gorilla at
the average distance $(1.6 + 1.7)/2 = 1.65$ (height 0.825), joins mouse and rat at 8.0 (height 4.0), and finally joins
the primate and rodent clusters at the average of the six primate–rodent distances, 30.5 (height 15.25). The resulting
tree, (((human, chimpanzee), gorilla), (mouse, rat)), is the accepted relationship among these species, and SciPy's
average-linkage clustering finds exactly the same merge distances. UPGMA assumes a **molecular clock** (all lineages
evolve at the same rate, so the root is equidistant from all leaves); when rates differ, methods such as neighbour
joining or likelihood-based searches over trees (Module 05: there are $(2n-3)!!$ of them) are used instead.

### 10.3 From graphs to trees: spanning trees

The process of extracting a tree from a graph is called finding a **spanning tree** (studied much more in algorithms
courses, such as CMU 02-613). For a connected graph $G = \langle V, E\rangle$, find a subgraph $T = \langle V, E'\rangle$ with

$$\text{(1) } E'\subseteq E, \qquad \text{(2) } |E'| = |V| - 1, \qquad \text{(3) } T \text{ is connected}.$$

*In words:* keep **all** the nodes but only some of the edges — just enough ($|V|-1$) to stay connected; by Section 9,
$T$ is then a tree. BFS gives one immediately: the edges from every node to its BFS parent. If the edges carry weights,
a **minimum spanning tree (MST)** has the smallest total weight; **Kruskal's algorithm** finds one by scanning the edges
from lightest to heaviest and keeping an edge whenever it joins two different components (it never creates a cycle). By
**Cayley's formula**, the complete graph $K_n$ has $n^{n-2}$ different spanning trees.

In [ ]:
# ── 10.3 A BFS spanning tree of the connected example graph ──
dist_bfs, parent_bfs = bfs("a", adjacency_lists(nodes6, left_edges))
bfs_tree = [(parent_bfs[v], v) for v in nodes6 if parent_bfs[v] is not None]
print("BFS tree edges:", bfs_tree)
print("(1) E' ⊆ E?", as_undirected(bfs_tree) <= as_undirected(left_edges), "| (2) |E'| = |V| − 1?",
      len(bfs_tree) == len(nodes6) - 1, "| (3) connected?", len(connected_components(nodes6, bfs_tree)) == 1,
      "| a tree?", is_tree(nodes6, bfs_tree))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.0))
draw_graph(left_edges, pos=conn_pos, ax=axes[0], title=f"G: {len(left_edges)} edges")
draw_graph(bfs_tree, nodes=nodes6, pos=conn_pos, ax=axes[1], title=f"BFS spanning tree: {len(bfs_tree)} edges")
plt.tight_layout(); plt.show()

🔍 **What just happened?** BFS from $a$ reached every node and recorded one parent per node; those 5 parent edges keep all
6 nodes, are edges of $G$, number $|V| - 1 = 5$, and connect everything — a spanning tree. Of the 7 edges of $G$, the two
dropped ones ($\{c, d\}$ and $\{e, f\}$) each closed a cycle. A BFS tree also contains a shortest path from the root to
every node.

In [ ]:
# ── 10.4 Kruskal's minimum spanning tree on the five species; brute force over all spanning trees; single linkage ──
def kruskal(nodes, weighted_edges) -> list:
    """Minimum spanning tree: take edges from lightest to heaviest, keep one if it joins two components."""
    parent = {v: v for v in nodes}

    def root(v):
        while parent[v] != v:
            v = parent[v]
        return v

    tree = []
    for w, u, v in sorted(weighted_edges):
        ru, rv = root(u), root(v)
        if ru != rv:
            parent[ru] = rv
            tree.append((u, v, w))
    return tree


species_edges = [(float(D_SPECIES[i, j]), SPECIES[i], SPECIES[j]) for i, j in itertools.combinations(range(5), 2)]
mst = kruskal(SPECIES, species_edges)
print("MST edges:", [(u, v, w) for u, v, w in mst], "| total weight", round(sum(w for _, _, w in mst), 3))

spanning = [subset for subset in itertools.combinations(species_edges, 4)
            if is_tree(SPECIES, [(u, v) for _, u, v in subset])]
best = min(sum(w for w, _, _ in s) for s in spanning)
print(f"all 4-edge subsets of the 10 edges: {math.comb(10, 4)}; spanning trees among them: {len(spanning)} "
      f"(Cayley: 5^3 = {5 ** 3}); minimum total weight by brute force: {best:.3f}")

heaviest = max(mst, key=lambda e: e[2])
clusters_single = connected_components(SPECIES, [(u, v) for u, v, w in mst if (u, v, w) != heaviest])
print("cut the heaviest MST edge", heaviest[:2], "→ single-linkage clusters:", clusters_single)
Z_single = hierarchy.linkage(squareform(D_SPECIES), method="single")
print("SciPy single-linkage merge heights:", Z_single[:, 2], "= sorted MST edge weights:", sorted(w for _, _, w in mst))

🔍 **What just happened?** Kruskal keeps human–chimpanzee (1.2), human–gorilla (1.6), skips chimpanzee–gorilla (1.7,
which would close a triangle), keeps mouse–rat (8.0) and human–mouse (30.0): total weight 40.8. Checking all
$\binom{10}{4} = 210$ four-edge subsets finds exactly $125 = 5^{5-2}$ spanning trees (Cayley's formula) and confirms that
the smallest total weight is 40.8 — Kruskal's greedy choices were optimal, as theory guarantees. Cutting the heaviest
MST edge splits the species into {chimpanzee, gorilla, human} and {mouse, rat}, and SciPy's single-linkage clustering
merges at exactly the MST edge weights 1.2, 1.6, 8.0, 30.0.

### 🌉 Bridge — single-linkage clustering is a minimum spanning tree

Single-linkage hierarchical clustering (BiologyToML [Module 12](../../../../ml/tutorial/BiologyToML/notebooks/12_clustering_kmeans_hierarchical.ipynb)) merges the
two clusters whose *closest* members are nearest. Its merge heights are exactly the edge weights of a minimum spanning
tree, and cutting the $K-1$ heaviest MST edges gives the $K$ single-linkage clusters (Gower & Ross 1969). So the
"chaining" weakness of single linkage is a property of MSTs: one chain of close neighbours is enough to join two groups.
The same objects — trees, distances, linkage — return in phylogenetics (UPGMA = average linkage), in single-cell
trajectory inference (an MST through cluster centres orders cells along a differentiation path, BiologyToML
[Module 32](../../../../ml/tutorial/BiologyToML/notebooks/32_cell_dynamics_velocity.ipynb)) and in decision trees ([Module 10](../../../../ml/tutorial/BiologyToML/notebooks/10_decision_trees_random_forests.ipynb)).

✏️ **Your turn.** Change the mouse–rat distance in `D_SPECIES` (both entries) from 8.0 to 40.0, so the two rodents are
farther apart than primates and rodents. Re-run 10.2 and 10.4. How do the UPGMA tree and the MST change? Is the UPGMA
result still biologically sensible — and what assumption of UPGMA does this "data" violate?

### 📘 Lecture notes — phylogenies and spanning trees

**Phylogeny.** A tree showing relatedness of species/individuals/strains: leaves = known examples (extant species,
sequenced strains), internal nodes = (probably unknown) ancestors; edge weights encode evolutionary distance; more
closely related species have a common ancestor farther from the root (lower). Human and chimpanzee share an ancestor and
only meet the mouse at the root.

**Building trees.** UPGMA = average-linkage clustering of a distance matrix (merge closest clusters, average distances,
heights $d/2$; assumes a molecular clock). The number of possible rooted trees, $(2n-3)!!$, rules out exhaustive search.

**Spanning tree.** For $G = \langle V, E\rangle$ find $T = \langle V, E'\rangle$ with (1) $E'\subseteq E$, (2)
$|E'| = |V| - 1$, (3) $T$ connected. BFS gives one; Kruskal gives a minimum-weight one; $K_n$ has $n^{n-2}$ spanning trees.

**Common confusion.** Condition (2) is about the *tree's* edges, $|E'| = |V| - 1$ — not $|E|$, the number of edges of the
original graph.

> 🎓 **Exam tip.** Be able to run UPGMA on a 4–5 taxon matrix by hand (keep track of cluster sizes in the averages) and to
> list the edges Kruskal picks, saying which edge is skipped because it would close a cycle.

---
## 11 · Graphs as matrices: a preview

> 📍 **Where we are.** Everything so far stored a graph as sets of pairs. Linear algebra (Modules 07–13) stores it as a
> table of numbers — the **adjacency matrix** — and then matrix arithmetic answers graph questions.

For nodes $v_1, \dots, v_n$, the adjacency matrix $A$ is the $n\times n$ table with

$$A_{ij} = \text{number of edges from } v_i \text{ to } v_j .$$

*In words:* row $i$ lists where node $i$'s edges go, column $j$ lists where node $j$'s edges come from; for a simple
directed graph every entry is 0 or 1. (It is the "relation as a grid of filled cells" picture of Module 04.) Consequences:
the **row sums are the out-degrees**, the **column sums are the in-degrees**, and for an undirected graph $A$ is
**symmetric** ($A_{ij} = A_{ji}$) with row sums equal to the degrees (a loop adds 1 to the diagonal entry in the toolkit's
convention, so the degree is the row sum plus the diagonal entry).

The real power appears when we multiply: the entry $(i, j)$ of $A^2 = A\cdot A$ (Module 08 defines the product) is

$$(A^2)_{ij} = \sum_{k=1}^{n} A_{ik}A_{kj},$$

*in words:* the number of intermediate nodes $k$ with $v_i\to v_k\to v_j$ — the number of **walks of length 2** from $v_i$
to $v_j$; likewise $A^\ell$ counts walks of length $\ell$ ($\ell$ edges, as in Section 3). (A *walk* may repeat nodes, and so may a
path in our Section 3 sense; textbooks that reserve "path" for walks without repeated nodes would call these walks.)

In [ ]:
# ── 11.1 The adjacency matrix of the inflammation network; degrees from sums; A² counts 2-step walks ──
A = adjacency_matrix(NET_NODES, NET_EDGES, directed=True)
print("A is", A.shape, "with", A.sum(), "ones = |E|")
print("row sums = out-degrees?", np.array_equal(A.sum(axis=1), deg_table["out-degree"].to_numpy()),
      "| column sums = in-degrees?", np.array_equal(A.sum(axis=0), deg_table["in-degree"].to_numpy()))
idx = {g: i for i, g in enumerate(NET_NODES)}
A2 = A @ A
for u, v in [("NFKB1", "NFKB1"), ("STAT1", "ISG15"), ("IL6", "SAA1"), ("NFKB1", "STAT3")]:
    by_hand = sum(A[idx[u], k] * A[k, idx[v]] for k in range(len(NET_NODES)))
    via = [NET_NODES[k] for k in range(len(NET_NODES)) if A[idx[u], k] and A[k, idx[v]]]
    print(f"(A²)[{u}, {v}] = {A2[idx[u], idx[v]]} (by the sum: {by_hand}); intermediate genes: {via}")
undirected_pairs = [tuple(sorted(e)) for e in as_undirected(NET_EDGES)]    # each feedback pair listed once
U = adjacency_matrix(NET_NODES, undirected_pairs, directed=False)
print(f"undirected version: {len(undirected_pairs)} edges, symmetric? {np.array_equal(U, U.T)}, row sums = degrees? "
      f"{all(U[idx[g]].sum() == degree(g, NET_EDGES) for g in NET_NODES)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, M, title in [(axes[0], A, "A: edge i → j"), (axes[1], A2, "A²: number of 2-step walks i → k → j")]:
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(NET_NODES))); ax.set_xticklabels(NET_NODES, rotation=90, fontsize=6)
    ax.set_yticks(range(len(NET_NODES))); ax.set_yticklabels(NET_NODES, fontsize=6)
    for i, j in zip(*np.nonzero(M)):
        ax.text(j, i, M[i, j], ha="center", va="center", fontsize=6, color="white" if M[i, j] == M.max() else "black")
    ax.set_xlabel("to (column j)"); ax.set_ylabel("from (row i)"); ax.set_title(title)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 14 × 14 matrix contains 15 ones — one per edge — and its row and column sums reproduce the
out- and in-degree table of Section 5. $(A^2)_{\text{NFKB1},\text{NFKB1}} = 2$: there are two walks of length 2 from NF-κB
back to itself, through TNF and through IL1B — the two feedback loops, found by arithmetic. STAT1 reaches ISG15 in two
steps in one way (via IRF1: the feed-forward loop), IL6 reaches SAA1 via STAT3, and NFKB1 reaches STAT3 via IL6; the
explicit sum $\sum_k A_{ik}A_{kj}$ gives the same numbers as `A @ A`.
In the undirected version the two arrows of each feedback loop become one edge (13 edges), and the matrix is
symmetric with row sums equal to the degrees. Module 08 explains matrix multiplication properly and computes $A^3$ for this network; Modules 12 and 20 use powers of
matrices for Markov chains.

### 🌉 Bridge — de Bruijn graphs: assembling a genome from k-mers

Module 04 cut DNA into k-mers. Genome assemblers glue them back with a **de Bruijn graph**: its nodes are the
$(k-1)$-mers, and every k-mer $x_1x_2\dots x_k$ becomes an edge from its prefix $x_1\dots x_{k-1}$ to its suffix
$x_2\dots x_k$. A sequence corresponds to a walk that uses **every edge exactly once** (an **Eulerian path**), and the
sequence is the walk's label: the first node followed by the last letter of every further node (Pevzner, Tang & Waterman
2001; Compeau, Pevzner & Tesler 2011).

In [ ]:
# ── 11.2 A de Bruijn graph from the 3-mers of a short sequence; all Eulerian paths = all consistent assemblies ──
def de_bruijn_edges(seq: str, k: int) -> list:
    """One edge prefix → suffix per occurrence of every k-mer (counted with the toolkit's kmer_counts)."""
    return [(kmer[:-1], kmer[1:]) for kmer, count in kmer_counts(seq, k).items() for _ in range(count)]


def eulerian_sequences(edges) -> set:
    """Spell every walk that starts at the node with one more out- than in-edge and uses each edge exactly once."""
    out_deg, in_deg = Counter(u for u, _ in edges), Counter(v for _, v in edges)
    starts = [v for v in set(out_deg) | set(in_deg) if out_deg[v] - in_deg[v] == 1] or [edges[0][0]]
    found = set()

    def extend(path, remaining):
        if not remaining:
            found.add(path[0] + "".join(node[-1] for node in path[1:]))
            return
        for i, (u, v) in enumerate(remaining):
            if u == path[-1]:
                extend(path + [v], remaining[:i] + remaining[i + 1:])

    extend([starts[0]], list(edges))
    return found


read_dna = "ATGGCGTGCA"
db_edges = de_bruijn_edges(read_dna, 3)
assemblies = eulerian_sequences(db_edges)
print("3-mers:", list(kmer_counts(read_dna, 3)), "→", len(db_edges), "edges")
print("sequences spelled by Eulerian paths:", sorted(assemblies))
print("all have the same 3-mers as the original?", all(kmer_counts(s, 3) == kmer_counts(read_dna, 3) for s in assemblies))
longer = simulate_dna(40, seed=11)
for k in [4, 6, 8]:
    sols = eulerian_sequences(de_bruijn_edges(longer, k))
    print(f"SIMULATED 40-mer, k = {k}: {len(sols)} consistent assembl{'y' if len(sols) == 1 else 'ies'}; "
          f"original recovered: {longer in sols}")

draw_graph(db_edges, directed=True, edge_labels={(u, v): u + v[-1] for u, v in db_edges},
           title="de Bruijn graph of the 3-mers of ATGGCGTGCA (edge label = 3-mer)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 8 overlapping 3-mers of ATGGCGTGCA give 8 edges between seven 2-mers. Two different walks
use every edge once, spelling ATGCGTGGCA and the original ATGGCGTGCA: both contain exactly the same 3-mers, so the k-mers
alone cannot decide between them — the repeated 2-mers TG and GC act like short **repeats** that make assembly ambiguous.
For the SIMULATED 40-base sequence, $k = 4$ allows 11 different assemblies, whereas with $k = 6$ and $k = 8$ every
$(k-1)$-mer occurs once and the reconstruction is unique — and the original is always among the solutions. This is the
graph-theoretic side of Module 04's lesson: once $4^{k-1}$ is large compared with the genome, k-mers become unique and
the de Bruijn graph untangles — real assemblers choose $k$ (and read lengths) with exactly this trade-off in mind.

### 🌉 Bridge — Bayesian networks and graph neural networks

* **Bayesian networks** (Module 20; BiologyToML [Modules 15–17](../../../../ml/tutorial/BiologyToML/notebooks/15_bayesian_networks_representation.ipynb)) attach a
  probability table $p(x_v\mid\text{parents})$ to every node of a directed acyclic graph; the parents of $v$ are its
  in-neighbours $N^{\text{in}}(v)$, and the joint distribution factorises along the edges, as in
  $p(a, b, c) = p(a)\,p(b\mid a)\,p(c\mid a, b)$ (Section 2).
* **Graph neural networks** (BiologyToML [Module 28](../../../../ml/tutorial/BiologyToML/notebooks/28_graph_neural_networks.ipynb)) update every node's feature
  vector from its neighbours' — a sum or average over $N(v)$, which in matrix form is a product with $A$. The next cell
  shows the simplest such "message passing": a signal entering at TNF spreads along the arrows by repeated multiplication
  with $A^\top$.

In [ ]:
# ── 11.3 Message passing: a signal at TNF spreads along the arrows, h_(t+1) = Aᵀ h_t ──
h = np.zeros(len(NET_NODES), dtype=int)
h[idx["TNF"]] = 1                                              # one unit of signal at TNF
for step in range(1, 5):
    h = A.T @ h                                                # node j collects Σ_i A[i, j]·h[i] from its in-neighbours
    print(f"after step {step}: " + ", ".join(f"{NET_NODES[j]}: {h[j]}" for j in np.nonzero(h)[0]))

🔍 **What just happened?** One unit of signal placed on TNF reaches NFKB1 after one step; after two steps NF-κB passes
it to its five targets (including TNF itself — the feedback loop); after three steps the signal arrives back at NFKB1
twice (via TNF and IL1B) and at STAT3 (via IL6); after four steps it has reached the acute-phase genes. Each number
counts the walks of that length from TNF — the entries of the TNF row of $A^t$. A graph neural network does the same
aggregation with learned weights and vectors of features instead of single numbers.

✏️ **Your turn.** (a) Start the signal at STAT1 instead. After how many steps does it stop spreading, and why? (b) Compute
`np.linalg.matrix_power(A, 3)[idx["NFKB1"]]` and explain the entry for NFKB1 itself.

## 📌 Key takeaways

- A graph is a tuple $G = \langle V, E\rangle$: directed edges are ordered pairs ($E\subseteq V\times V$), undirected
  edges are 2-element sets; $u = v$ gives a self-loop. Binding is undirected (PPI), regulation is directed (GRN).
- A path $\langle v_1, \dots, v_k\rangle$ follows edges; its length is $k - 1$ edges (state the convention). Labels
  $\ell: E\to\Sigma^*$ concatenate along paths, weights $w: E\to\mathbb R$ add up.
- $N_G(v)$ is the set of neighbours; $\operatorname{degree}(v) = |N(v)| + \mathbb 1(v\in N(v))$ (loops count twice);
  $\sum_v\deg(v) = 2|E|$. Directed graphs have in- and out-neighbourhoods and degrees, $\sum\text{in} = \sum\text{out} = |E|$.
- Regular graphs have one common degree; $K_n$ is $(n-1)$-regular with $\binom n2$ edges; a clique is a complete subgraph.
- Subgraphs keep some nodes and some of their edges; induced subgraphs keep all edges among the chosen nodes — and
  depend critically on the node list (hidden regulators such as NF-κB).
- Bipartite graphs have two groups with edges only between them (metabolites ↔ reactions); a graph is bipartite iff it
  has no odd cycle, which a two-colouring BFS tests.
- Connected: $\forall u, v\ \exists$ a path. Components are *maximal* connected subgraphs; BFS finds them and shortest
  path lengths in $O(|V| + |E|)$.
- Trees are connected with $|V| - 1$ edges (hence acyclic, with unique paths); leaves have degree 1 (no children in
  rooted trees); full $k$-ary trees have $ki + 1$ nodes. Phylogenies are rooted, weighted trees; UPGMA builds one from
  distances; spanning trees keep all nodes with $|V| - 1$ edges, and the MST is single-linkage clustering.
- The adjacency matrix turns graphs into linear algebra: row/column sums are degrees and $A^2$ counts 2-step walks.

---
## ✅ Checkpoint

1. Write $G = \langle V, E\rangle$ for the undirected graph on $\{x, y, z\}$ with the edges $x$–$y$, $y$–$z$, $x$–$z$ and a self-loop at $x$. Give all degrees and check the handshake lemma.
   <details><summary>Answer</summary>$V = \{x, y, z\}$, $E = \{\{x,y\}, \{y,z\}, \{x,z\}, \{x\}\}$ (the loop $\{x, x\} = \{x\}$). $N(x) = \{x, y, z\}$, so $\deg(x) = 3 + 1 = 4$; $\deg(y) = \deg(z) = 2$. Sum $8 = 2\cdot 4$ edges.</details>
2. Which of these are naturally directed, and which undirected: protein–protein binding, transcriptional regulation, "is a sibling of", a food web, a phylogeny drawn from the root?
   <details><summary>Answer</summary>Undirected: protein binding, siblings. Directed: regulation (TF → target), food web (prey → predator, energy flow), a rooted phylogeny (ancestor → descendant). (An unrooted phylogeny is undirected.)</details>
3. In the course-schedule graph, what is the length of the path ⟨Math, Physics, Chemistry, Art⟩? Is ⟨Math, History, Art⟩ a path?
   <details><summary>Answer</summary>3 edges (4 nodes; some texts would say length 4 — state the convention). ⟨Math, History, Art⟩ is not a path: there is no edge History → Art (History goes to Economics).</details>
4. For a route with edge labels "Main St", "Broadway", "5th Ave" and weights 1.2, 0.8, 2.5 km, what are the path label and path weight?
   <details><summary>Answer</summary>Label = the concatenation "Main St · Broadway · 5th Ave" (an element of $\Sigma^*$); weight = $1.2 + 0.8 + 2.5 = 4.5$ km.</details>
5. Prove that every undirected graph has an even number of nodes of odd degree.
   <details><summary>Answer</summary>$\sum_v\deg(v) = 2|E|$ is even. The sum of the even degrees is even, so the sum of the odd degrees must be even too, which requires an even number of odd terms.</details>
6. In the A–H directed example, give $N^{\text{in}}(E)$, $N^{\text{out}}(E)$ and $N(E)$. Which node has in-degree 0?
   <details><summary>Answer</summary>$N^{\text{in}}(E) = \{B, C, F\}$, $N^{\text{out}}(E) = \{B, F\}$, $N(E) = \{B, C, F\}$ (the union). A has in-degree 0 (and G out-degree 0).</details>
7. How many edges does $K_7$ have, and what is its degree? Does the graph with edges AB, AC, BC, CD contain a 4-node clique?
   <details><summary>Answer</summary>$\binom72 = 21$ edges; 6-regular. No: a 4-clique would need all 6 pairs among A, B, C, D, but AD and BD are missing; the largest clique is {A, B, C}.</details>
8. For the same graph, how many subgraphs have exactly the node set {A, C, D}, and which of them is induced?
   <details><summary>Answer</summary>The candidate edges among A, C, D are AC and CD, so $2^2 = 4$ subgraphs (no edges, AC only, CD only, both); the induced one keeps both AC and CD.</details>
9. Is a cycle with 5 nodes bipartite? With 6 nodes? Why?
   <details><summary>Answer</summary>The 6-cycle is bipartite (colour alternately); the 5-cycle is not — colours must alternate around a cycle, impossible with an odd number of edges. A graph is bipartite iff it has no odd cycle.</details>
10. A tree has 20 nodes. How many edges does it have? A connected graph with 20 nodes has 20 edges — is it a tree?
    <details><summary>Answer</summary>19 edges. No: a connected graph with $|V|$ edges has one edge too many; it contains exactly one cycle.</details>
11. A full binary tree has 7 internal nodes. How many nodes and leaves does it have? In the 13-node rooted example, what are the children of B and the leaves?
    <details><summary>Answer</summary>$n = 2\cdot 7 + 1 = 15$ nodes, $15 - 7 = 8$ leaves. Children of B: {D, E, F}; leaves {D, F, H, J, K, L, M}.</details>
12. State the three conditions of a spanning tree. For the five species, which edge does Kruskal skip first, and what is the MST's relation to single-linkage clustering?
    <details><summary>Answer</summary>$E'\subseteq E$, $|E'| = |V| - 1$, $T$ connected. Kruskal skips chimpanzee–gorilla (1.7) because human–chimpanzee and human–gorilla already connect those three. Single-linkage merge heights equal the MST edge weights; cutting the heaviest MST edges gives the single-linkage clusters.</details>
13. What does the entry $(A^2)_{ij}$ of the adjacency matrix count? Why is $(A^2)_{\text{NFKB1},\text{NFKB1}} = 2$?
    <details><summary>Answer</summary>The number of walks of length 2 from $v_i$ to $v_j$ (intermediate nodes $k$ with $i\to k\to j$). NF-κB → TNF → NF-κB and NF-κB → IL1B → NF-κB: the two feedback loops.</details>

## 🧠 Questions & discussion

1. A PPI database lists 300,000 interactions among 20,000 human proteins. What is the mean degree? Why might the observed hubs partly reflect *study bias* (well-studied proteins are tested more often) rather than biology?
2. The induced subgraph of differentially expressed genes lost the regulators NF-κB and STAT3. Propose two ways to recover "hidden" regulators from data, and discuss what each assumes.
3. Co-expression networks connect correlated genes; regulatory networks connect regulators to targets. Why can a co-expression edge exist without any direct regulation (think of a common regulator, Section 2's common-cause graph)? Which graph type — directed or undirected — fits each network?
4. UPGMA assumes all lineages evolve at the same rate. Give a biological situation in which this "molecular clock" fails badly (e.g. a fast-evolving parasite or endosymbiont), and predict what UPGMA would do wrong.
5. Directed cycles (feedback loops) are common in regulatory networks but forbidden in Bayesian networks (which must be acyclic). How could you model a feedback loop over time with an acyclic graph? (Hint: unroll it — a dynamic Bayesian network.)

## 🚀 Bonus — going further

- **Shortest weighted paths.** BFS finds shortest paths by number of edges. Implement Dijkstra's algorithm (a priority queue with `heapq`) for the route graph of Section 3 and check the shortest LA → Orlando route.
- **Eulerian paths in general.** Prove that a connected directed graph has an Eulerian path iff at most one node has out-degree − in-degree = 1, at most one has in-degree − out-degree = 1 and all others are balanced. Check the condition on the de Bruijn graph of 11.2.
- **Counting spanning trees with a matrix.** Kirchhoff's matrix-tree theorem: the number of spanning trees equals any cofactor of the graph Laplacian $L = D - A$ (degree matrix minus adjacency). Verify it gives $125 = 5^3$ for $K_5$ after Module 08 (determinants).
- **Neighbour joining.** Read about the neighbour-joining algorithm (Saitou & Nei 1987), which does not assume a molecular clock, and compare its tree with UPGMA on the modified matrix of the ✏️ exercise in Section 10.
- **Further reading.** Liben-Nowell, *Connecting Discrete Mathematics and Computer Science*, Ch. 11 (graphs and trees); Rosen, *Discrete Mathematics and Its Applications*, Ch. 10–11; Alon, *An Introduction to Systems Biology* (network motifs such as the feed-forward loop).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            06 · Graphs and trees
Input:             illustrative inflammation network (14 genes, 15 edges); SIMULATED infection expression
                   (24 × 400); typed-in small graphs (degree exercises, A–H, bipartite, glycolysis, PPI, species
                   distances); SIMULATED random/preferential-attachment networks and DNA for a de Bruijn graph
Output:            neighbourhoods and degrees, in/out-degree tables, cliques, (induced) subgraphs, two-colourings,
                   connected components, BFS distances, tree properties, a UPGMA tree, BFS and minimum spanning
                   trees, adjacency matrices and A², Eulerian assemblies
Method:            set operations on edge lists; BFS (deque); union–find; Kruskal; UPGMA (checked with SciPy)
Key equation:      degree(v) = |N(v)| + 1(v ∈ N(v));  Σ_v deg(v) = 2|E|;  tree ⇔ connected ∧ |E| = |V| − 1
Key parameters:    which node list induces the subgraph; correlation threshold of the co-expression network;
                   k of the de Bruijn graph; edge weights / distances
What changed when I modified …  (e.g. liver instead of spleen in 6.2; threshold 0.6 in 8.3; D_SPECIES in 10.2)
One failure mode:  an induced subgraph on an incomplete node list hides regulators (NFKB1, STAT3);
                   counting a self-loop once breaks the handshake lemma
Connection to 02-680: graphs ⟨V, E⟩, paths, labels/weights, degree, regular/complete graphs, subgraphs,
                   bipartite graphs, connectivity, trees, rooted and k-ary trees, phylogeny, spanning trees
```

## 📚 References

- Liben-Nowell D. (2022) *Connecting Discrete Mathematics and Computer Science*, 2nd ed. Cambridge University Press — §11.2–11.4 (graphs, paths and connectivity, trees); the source of many examples here (route graph, neighbourhood questions, the A–H examples, rooted trees).
- Rosen K. H. (2019) *Discrete Mathematics and Its Applications*, 8th ed. McGraw-Hill — §10.2 (degrees, the handshake theorem, the degree examples), §11.1 (trees, full m-ary trees T1–T4).
- Bishop C. M. (2006) *Pattern Recognition and Machine Learning*. Springer — §8.1 (directed graphical models) and §8.4 (trees and polytrees).
- Wasserman L. (2004) *All of Statistics*. Springer — Ch. 17 (directed graphs: adjacent, parent, child, directed path).
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Barabási A.-L., Albert R. (1999) Emergence of scaling in random networks. *Science* 286:509–512.
- Jeong H., Mason S. P., Barabási A.-L., Oltvai Z. N. (2001) Lethality and centrality in protein networks. *Nature* 411:41–42.
- Milo R. *et al.* (2002) Network motifs: simple building blocks of complex networks. *Science* 298:824–827.
- Sokal R. R., Michener C. D. (1958) A statistical method for evaluating systematic relationships. *University of Kansas Science Bulletin* 38:1409–1438 (UPGMA).
- Kruskal J. B. (1956) On the shortest spanning subtree of a graph and the traveling salesman problem. *Proceedings of the American Mathematical Society* 7:48–50.
- Gower J. C., Ross G. J. S. (1969) Minimum spanning trees and single linkage cluster analysis. *Journal of the Royal Statistical Society C (Applied Statistics)* 18:54–64.
- Pevzner P. A., Tang H., Waterman M. S. (2001) An Eulerian path approach to DNA fragment assembly. *PNAS* 98:9748–9753.
- Compeau P. E. C., Pevzner P. A., Tesler G. (2011) How to apply de Bruijn graphs to genome assembly. *Nature Biotechnology* 29:987–991.
- Felsenstein J. (2004) *Inferring Phylogenies*. Sinauer (phylogenies, distance methods, counting trees).

## ➡️ Next

**[Module 07 · ➡️ Vectors: profiles, lengths and angles](07_vectors.ipynb)** — Part C begins. A vector of $\mathbb R^n$ is an $n$-tuple of real numbers (Module 04), and every row
of the adjacency matrix you just built is one: the "profile" of a gene's outgoing edges. The next module gives such
tuples a geometry — addition, lengths, angles, distances — and uses it on the expression profiles of the infection
experiment. Then [Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb) multiplies matrices properly and computes $A^2$ and $A^3$ of the inflammation network,
counting the regulatory walks you met in Section 11.